# Janus 00A｜個股資料獨立收檔 v3.8.4（Colab）

沿用已穩定的 v3.8.2 個股／OHLCV／營收／財報／估值收集核心；**所有原始程式 code cells 保持不變**，可以單獨執行全部儲存格。
不需要執行 00B 大盤／產業收檔。大盤／產業已改由另一 Notebook 採集。
本 Notebook 不跑模型、貼標或發布 Risk ACTIVE。


## 1．安裝與掛載 Drive

In [ ]:
import sys,subprocess,os,json,hashlib,importlib
from pathlib import Path
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'requests==2.32.5','numpy==2.2.6','pandas==2.2.3','pyarrow==20.0.0',
    'lxml==5.4.0','beautifulsoup4==4.13.4','yfinance==0.2.66'])
from google.colab import drive
drive.mount('/content/drive')
PROJECT=Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
ROOT=PROJECT/'資料源/JanusDataset_v3_202610'
if not (ROOT/'data').is_dir():raise FileNotFoundError('找不到既有資料根目錄：'+str(ROOT))


## 2．長期設定
**預設不需改日期**：台灣時間每次自動取「上個完整月份」。若 2027 年 3 月執行，最晚封存 2027-02；3 月須等 4 月再收。

可選擇 `END_MONTH_OVERRIDE='2027-02'` 做歷史有界重跑（不得設定到本月或未來）。舊資料與已收來源照常保留。


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
from pathlib import Path
# Earliest research history is retained indefinitely; future runs NEVER truncate
# previous official inputs when 36/48 month rolling lookback advances.
START_MONTH='2025-01'                   # OHLCV research history start
REVENUE_FIRST_MONTH='2022-10'
VALUATION_FIRST_MONTH='2023-10'
FINANCIAL_FIRST_QUARTER='2022Q4'
END_MONTH_OVERRIDE=None                # e.g. '2027-02'; None = last closed month in Taiwan
as_of=datetime.now(ZoneInfo('Asia/Taipei')).date()
last_complete_month=pd.Period(as_of, freq='M')-1
END_MONTH=str(last_complete_month if END_MONTH_OVERRIDE is None else pd.Period(END_MONTH_OVERRIDE,'M'))
if pd.Period(END_MONTH,'M')>last_complete_month:
    raise ValueError(f'禁止封存尚未結束的月份：{END_MONTH}；最近可封存 {last_complete_month}')
if pd.Period(END_MONTH,'M')<pd.Period(START_MONTH,'M'):
    raise ValueError('END_MONTH 不得早於 START_MONTH')
OHLCV_MAX_REQUESTS=30
OHLCV_MAX_MINUTES=12
OHLCV_INTERVAL_SECONDS=2
TPEx_VALUATION_REPAIR=True        # 優先採櫃買中心歷史日期 HTML 整市場表格
OFFICIAL_MAX_REQUESTS=160
OFFICIAL_MAX_MINUTES=45
OFFICIAL_INTERVAL_SECONDS=3
ENABLE_FINMIND_FALLBACK=False
FINMIND_MAX_REQUESTS=20
FINMIND_MAX_MINUTES=10
FINMIND_INTERVAL_SECONDS=15
RETRY_EMPTY_FINMIND=False
ENABLE_YAHOO_FALLBACK=False
YAHOO_MAX_SYMBOLS=20
VERIFY_ALL_PARQUET_SHA256=True
FINMIND_TOKEN=os.environ.get('FINMIND_TOKEN','')
try:
    from google.colab import userdata
    FINMIND_TOKEN=userdata.get('FINMIND_TOKEN') or FINMIND_TOKEN
except Exception:
    pass
print(f'台灣執行日：{as_of}｜本輪有效期間 {START_MONTH} ～ {END_MONTH}；原始版本保留，僅補缺口')
print('v3.6 已啟用：2023-10 上市估值的官方歷史資格核查（非PIT認證）')
RECHECK_ONLY=False  # 只重驗現有 canonical/reports、不補收、不重建 Parquet 時改 True
# D07 四模型共用因子 C1 探查：只有明確改成兩個 True 才執行，不干擾原本的 v3.6 收集。
FACTOR_ONLY=False         # 預設啟用增量補收；True 僅略過收集/執行 C1 來源探查
FACTOR_SOURCE_PROBE=False  # True：啟用市場指數小批 raw/schema 探查（不是完整歷史補收）
FACTOR_PROBE_MAX_REQUESTS=4
if RECHECK_ONLY and FACTOR_ONLY:
    raise ValueError('RECHECK_ONLY 與 FACTOR_ONLY 不可同時啟用')



## 3．載入本檔內嵌程式
不需clone GitHub，也不需另外下載Python檔。

In [ ]:
CODE_DIR=Path('/content/janus_official_batch_v3')
CODE_DIR.mkdir(exist_ok=True)
SOURCES={'quality_gate_zh.py': '"""繁體中文研究資料驗收（資料可試訓 != 模型通過 OOS / 發布）。"""\nfrom __future__ import annotations\nimport hashlib, json\nfrom pathlib import Path\nfrom datetime import date\nimport pandas as pd\n\nTHRESHOLDS={\'twse\':0.90,\'tpex\':0.70}\nFINANCIAL={\'income\',\'balance\',\'cashflow\'}\nNAMES={\'ohlcv\':\'股價 OHLCV\',\'revenue\':\'月營收\',\'valuation\':\'PE／PB 估值\',\n       \'income\':\'損益表\',\'balance\':\'資產負債表\',\'cashflow\':\'現金流量表\',\n       \'corporate_actions\':\'公司行動\'}\n\n\ndef _quarter_published(period, as_of):\n    return (as_of-pd.Period(period,freq=\'Q\').end_time.date()).days>=50\n\n\ndef _due(kind,period,as_of):\n    if kind in FINANCIAL:\n        return _quarter_published(period,as_of)\n    if kind==\'revenue\':\n        return (as_of-pd.Period(period,freq=\'M\').end_time.date()).days>=15\n    return True\n\n\ndef _read_actual(release):\n    """Read model-facing Parquet, not stale upstream official-only coverage."""\n    observed={}\n    for kind in (\'ohlcv\',\'revenue\',\'valuation\',\'financials\'):\n        for path in sorted((Path(release)/kind).glob(\'*.parquet\')):\n            if kind==\'financials\':\n                cols=[\'symbol\',\'market\',\'dataset\',\'fiscal_quarter\']\n            elif kind==\'revenue\':\n                cols=[\'symbol\',\'market\',\'fiscal_month\']\n            else:\n                cols=[\'symbol\',\'market\', \'trade_date\']\n            frame=pd.read_parquet(path,columns=cols)\n            if frame.empty: continue\n            if kind==\'financials\':\n                frame=frame.rename(columns={\'dataset\':\'kind\',\'fiscal_quarter\':\'period\'})\n            elif kind==\'revenue\':\n                frame[\'kind\']=\'revenue\';frame=frame.rename(columns={\'fiscal_month\':\'period\'})\n            else:\n                frame[\'kind\']=kind;frame[\'period\']=frame.trade_date.astype(str).str[:7]\n            for (dataset, market, period), g in frame.groupby([\'kind\',\'market\',\'period\']):\n                if market not in THRESHOLDS or not period: continue\n                observed.setdefault((str(dataset),str(market),str(period)),set()).update(g[\'symbol\'].astype(str))\n    return observed\n\n\ndef _validate(release,manifest):\n    """Validate every published file hash and row count before PASS."""\n    issues=[]\n    for relative, expected in manifest.get(\'files\',{}).items():\n        p=Path(release)/relative\n        if not p.is_file(): issues.append(\'FILE_MISSING: \'+relative);continue\n        if hashlib.sha256(p.read_bytes()).hexdigest()!=expected[\'sha256\']:\n            issues.append(\'SHA256_MISMATCH: \'+relative)\n    return issues\n\n\ndef assess(root, release, manifest, roster, tasks, *, audit, batch, as_of=None,\n           validate_hashes=True):\n    root=Path(root);release=Path(release);as_of=as_of or date.today()\n    canonical_issues=_validate(release,manifest) if validate_hashes else []\n    pointer_path=root/\'canonical\'/\'CURRENT.json\'\n    try:\n        pointer=json.loads(pointer_path.read_text(\'utf-8\'))\n        if pointer.get(\'release\')!=release.name or pointer.get(\'format\')!=\'parquet\':\n            canonical_issues.append(\'CURRENT_POINTER_MISMATCH\')\n    except (OSError,ValueError):canonical_issues.append(\'CURRENT_POINTER_MISSING_OR_INVALID\')\n    observed=_read_actual(release)\n    from official_batch import period_market_roster, audited_eligible_roster\n    cohort=period_market_roster(root,roster,{t[\'period\'] for t in tasks})\n    market_rosters={market:{s for s,m in roster.items() if m[\'market\']==market}\n                    for market in THRESHOLDS}\n    old_errors={}\n    for e in batch.errors:\n        t=e.get(\'task\',{})\n        old_errors[(t.get(\'kind\'),t.get(\'market\'),t.get(\'period\'))]=e.get(\'error\')\n    rows=[]\n    for t in tasks:\n        kind,market,period=t[\'kind\'],t[\'market\'],t[\'period\']\n        raw_expected=cohort[period][market]\n        expected,eligibility=audited_eligible_roster(root,kind,market,period,raw_expected)\n        if eligibility is not None and (observed.get((kind,market,period),set()) & raw_expected - expected):\n            raise RuntimeError(\'HISTORICAL_ROSTER_AUDIT_OBSERVED_CONFLICT\')\n        present=observed.get((kind,market,period),set())&expected\n        pct=len(present)/len(expected) if expected else 0.0\n        pending=not _due(kind,period,as_of)\n        saved=batch.load(t) is not None\n        source_reason=old_errors.get((kind,market,period))\n        if pending:\n            state=\'PENDING_PUBLICATION\';reason=(\'尚在預留的財報公告期（期末後50天內）\' if kind in FINANCIAL else \'月營收公告預留期（期末後15天內）\')+\'，不算失敗\'\n        elif pct>=THRESHOLDS[market]:\n            state=\'PASS\';reason=(\'已依112檔官方歷史資格核查調整研究分母；非完整歷史PIT認證\'\n                                 if eligibility else \'已達研究覆蓋率門檻；但歷史母體/PIT仍未認證\')\n        elif not present:\n            state=\'BLOCKED\';reason=((\'官方錯誤：\'+source_reason) if source_reason else\n                                    (\'官方批次已存但尚無有效模型資料\' if saved else \'官方批次未成功保存／尚未取得資料\'))\n        else:\n            state=\'PARTIAL\';reason=\'部分股票未有可用值，含可能未上市/不適用；需逐股票核查\'\n        action=\'不需補收\' if state==\'PASS\' else (\'等待公告\' if pending else\n                 (\'先補官方歷史月末估值；官方失敗可試 FinMind TaiwanStockPER 歷史區間（Yahoo 當前 PE/PB 不可當歷史值）\'\n                   if kind==\'valuation\' else\n                  \'優先重試官方 MOPS 該季度；若失敗用 Yahoo 季報/FinMind 財報定向補缺\'\n                   if kind in FINANCIAL else\n                  \'僅補此資料集／月份／市場，核對母體與來源\'))\n        rows.append({\'資料類型\':NAMES.get(kind,kind),\'dataset\':kind,\'市場\':\'上市\' if market==\'twse\' else \'上櫃\',\n                     \'market\':market,\'月份／季度\':period,\'有效股票數\':len(present),\n                     \'母體股票數\':len(expected),\'覆蓋率\':round(pct*100,2),\n                     \'門檻\':int(100*THRESHOLDS[market]),\'官方批次已保存\':saved,\n                     \'狀態\':state,\'缺口原因\':reason,\'建議補收\':action,\n                     \'母體資格依據\': eligibility[\'basis\'] if eligibility else \'CURRENT_ROSTER_NOT_HISTORICAL_PIT\'})\n    # OHLCV market×month is itself an expected collection task, not a permanent\n    # 18-month shortcut. Every new month gets a visible status and gap reason.\n    for a in audit:\n        period=a[\'month\']\n        mk=a.get(\'markets\') or {}\n        for market in THRESHOLDS:\n            metrics=mk.get(market,{})\n            observed_count=int(metrics.get(\'observed_symbols\',0) or 0)\n            expected_count=int(metrics.get(\'snapshot_symbols\',len(market_rosters[market])) or 0)\n            symbol_ratio=float(metrics.get(\'observed_ratio\',0) or 0)\n            stock_day_ratio=metrics.get(\'stock_day_ratio\')\n            quality=a.get(\'status\')==\'VERIFIED_RESEARCH\' and bool(metrics.get(\'passes\',False))\n            note=(\'研究月檔與市場股票／股票交易日覆蓋率均通過\' if quality else\n                  (a.get(\'reason\') or (\'尚未完成此月份行情/交易日的封存\' if a.get(\'status\')==\'MISSING_OR_CHECKPOINT\' else \'來源月檔驗證不通過或覆蓋率不足\')))\n            rows.append({\'資料類型\':NAMES[\'ohlcv\'],\'dataset\':\'ohlcv\',\'市場\':\'上市\' if market==\'twse\' else \'上櫃\',\n                         \'market\':market,\'月份／季度\':period,\'有效股票數\':observed_count,\'母體股票數\':expected_count,\n                         \'覆蓋率\':round(100*symbol_ratio,2),\'股票交易日覆蓋率\':(round(100*float(stock_day_ratio),2) if stock_day_ratio is not None else None),\n                         \'門檻\':int(100*THRESHOLDS[market]),\'官方批次已保存\':quality,\'狀態\':\'PASS\' if quality else \'BLOCKED\',\n                         \'缺口原因\':note,\'建議補收\':\'不需補收\' if quality else \'沿用 checkpoint，僅補此市場/月的 OHLCV 交易日\'})\n    gap_df=pd.DataFrame(rows)\n\n    active=gap_df[gap_df[\'狀態\']!=\'PENDING_PUBLICATION\']\n    dataset_rows=[]\n    for kind in (\'ohlcv\',\'revenue\',\'valuation\',\'income\',\'balance\',\'cashflow\'):\n        g=active[active.dataset==kind]\n        if kind==\'ohlcv\':\n            valid=[a for a in audit if a.get(\'status\')==\'VERIFIED_RESEARCH\']\n            status=\'PASS\' if len(valid)==len(audit) and len(valid)>0 and not canonical_issues else \'PARTIAL\'\n            all_months=sorted({str(a.get(\'month\')) for a in audit})\n            span=f\'{all_months[0]}～{all_months[-1]}\' if all_months else \'-\'\n            dataset_rows.append({\'資料類型\':NAMES[kind],\'期間\':span,\'達標\':len(valid),\n                                 \'應驗\':len(audit),\'狀態\':status,\'備註\':\'逐月兩市場股票及交易日覆蓋率；非 PIT/公司行動認證\'})\n            continue\n        total=len(g); passed=int((g[\'狀態\']==\'PASS\').sum())\n        state=\'PASS\' if total and passed==total and not canonical_issues else \'PARTIAL\' if passed else \'BLOCKED\'\n        all_periods=gap_df[gap_df.dataset==kind][\'月份／季度\']\n        scope=(str(all_periods.min())+\'～\'+str(all_periods.max())) if len(all_periods) else \'-\'\n        dataset_rows.append({\'資料類型\':NAMES[kind],\'期間\':scope,\'達標\':passed,\n                             \'應驗\':total,\'狀態\':state,\'備註\':\'依上市90%／上櫃70%逐期判定；不含公告未屆期\'})\n    actions=int(manifest.get(\'tables\',{}).get(\'corporate_actions\',0))\n    dataset_rows.append({\'資料類型\':NAMES[\'corporate_actions\'],\'期間\':\'按有資料的月分區\',\n                          \'達標\':actions,\'應驗\':None,\'狀態\':\'PARTIAL\' if actions else \'BLOCKED\',\n                          \'備註\':\'公司行動已收錄，但拆股／減資調整的可比性尚未驗證；達標欄為筆數\'})\n    datasets=pd.DataFrame(dataset_rows)\n    def ds(k):return datasets.loc[datasets[\'資料類型\']==NAMES[k],\'狀態\'].iloc[0]\n    valid_ohlcv=len([a for a in audit if a.get(\'status\')==\'VERIFIED_RESEARCH\'])\n    has_ohlcv=valid_ohlcv>=18 and not canonical_issues\n    val_twse=active[(active.dataset==\'valuation\')&(active.market==\'twse\')]\n    twse_good=int((val_twse[\'狀態\']==\'PASS\').sum())\n    quant=\'PARTIAL\' if has_ohlcv else \'BLOCKED\'\n    risk=\'PASS\' if has_ohlcv else \'BLOCKED\'\n    fundamental=(\'PASS\' if ds(\'revenue\')==\'PASS\' and all(ds(k)==\'PASS\' for k in FINANCIAL)\n                 and manifest.get(\'feature_checks\',{}).get(\'financial_comparable\',False)\n                 else \'PARTIAL\' if ds(\'revenue\')==\'PASS\' else \'BLOCKED\')\n    valuation=\'PASS\' if ds(\'valuation\')==\'PASS\' and ds(\'income\')==\'PASS\' and ds(\'balance\')==\'PASS\' else (\'PARTIAL\' if twse_good>=24 and has_ohlcv else \'BLOCKED\')\n    def _span(kind, market=None):\n        part=gap_df[(gap_df.dataset==kind)&(gap_df[\'狀態\']==\'PASS\')]\n        if market: part=part[part.market==market]\n        periods=sorted(set(part[\'月份／季度\']))\n        return (f\'{periods[0]}～{periods[-1]}（{len(periods)}期達標）\'\n                if periods else \'尚無達標期別\')\n    ohlcv_periods=sorted({str(a.get(\'month\')) for a in audit if a.get(\'status\')==\'VERIFIED_RESEARCH\' and a.get(\'month\')})\n    ohlcv_span=(f\'{ohlcv_periods[0]}～{ohlcv_periods[-1]}（{len(ohlcv_periods)}個月）\'\n               if ohlcv_periods else f\'{valid_ohlcv}個驗證月（起訖未標示）\')\n    roles=pd.DataFrame([\n     {\'角色\':\'Risk\',\'狀態\':risk,\'可試訓\':risk==\'PASS\',\'可使用期間\':ohlcv_span,\n      \'研究範圍\':\'OHLCV 波動風險試訓\',\'仍需\': \'OOS、公司行動及 PIT 實證；不代表可以發布模型\'},\n     {\'角色\':\'Quant\',\'狀態\':quant,\'可試訓\':quant==\'PARTIAL\',\'可使用期間\':ohlcv_span,\n      \'研究範圍\':\'價量技術子模型\',\'仍需\':\'市場 benchmark 對齊、公司行動/成熟標籤；超額報酬子模型尚未放行\'},\n     {\'角色\':\'Fundamental\',\'狀態\':fundamental,\'可試訓\':fundamental==\'PARTIAL\',\'可使用期間\':\'月營收 \'+_span(\'revenue\'),\n      \'研究範圍\':\'月營收 YoY 子模型\',\'仍需\':\'季財報／EPS 可比性、首次公告 PIT 與修訂時間驗證\'},\n     {\'角色\':\'Valuation\',\'狀態\':valuation,\'可試訓\':valuation in (\'PASS\',\'PARTIAL\'),\'可使用期間\':\'上市估值 \'+_span(\'valuation\',\'twse\'),\n      \'研究範圍\':\'上市／上櫃估值分位數研究\',\'仍需\':\'歷史完整PIT名冊、財報/估值分組可比性與OOS；研究覆蓋率PASS不代表正式發布\'},\n     {\'角色\':\'Event\',\'狀態\':\'BLOCKED\',\'可試訓\':False,\'可使用期間\':\'未驗證人審標註資料\',\'研究範圍\':\'規則可用、監督式試訓未放行\',\'仍需\':\'事件人審標籤與修訂去重驗證\'},\n    ])\n    if canonical_issues:\n        roles.loc[:,\'狀態\']=\'BLOCKED\';roles.loc[:,\'可試訓\']=False\n    pending=gap_df[(gap_df[\'狀態\']!=\'PASS\')&(gap_df[\'狀態\']!=\'PENDING_PUBLICATION\')].copy()\n    recommendations=pending[[\'資料類型\',\'市場\',\'月份／季度\',\'覆蓋率\',\'缺口原因\',\'建議補收\']].copy()\n    summary=pd.DataFrame([\n      {\'階段\':\'Parquet 資料包整併\',\'狀態\':\'PASS\' if not canonical_issues else \'BLOCKED\',\n       \'說明\':\'CURRENT、各資料分區與 SHA256 檢查\' if not canonical_issues else \';\'.join(canonical_issues[:5])},\n      {\'階段\':\'官方／第三方缺口補收\',\'狀態\':\'PASS\' if pending.empty else \'PARTIAL\',\n       \'說明\':f\'尚有 {len(pending)} 個資料集×市場×期別低於門檻；公告未到期另列\'},\n      {\'階段\':\'研究試訓入口\',\'狀態\':\'PASS\' if (roles.loc[roles.角色==\'Risk\',\'狀態\'].iloc[0]==\'PASS\') else \'PARTIAL\',\n       \'說明\':\'以各角色狀態為準；不代表 OOS 或 production promotion\'},\n    ])\n    dest=root/\'reports\'; dest.mkdir(parents=True,exist_ok=True)\n    files=[(\'驗收總表.csv\',summary),(\'資料收集完成度.csv\',datasets),(\'模型試訓準入.csv\',roles),\n           (\'逐期缺口與原因.csv\',gap_df),(\'定向補收建議.csv\',recommendations)]\n    for filename,frame in files:\n        frame.to_csv(dest/filename,index=False,encoding=\'utf-8-sig\')\n    # Override old fixed-False placeholder with a real, per-role research readiness calculation.\n    roles.assign(role=roles[\'角色\'],training_ready=roles[\'可試訓\'],status=roles[\'狀態\'])[\n        [\'role\',\'training_ready\',\'status\',\'可使用期間\',\'研究範圍\',\'仍需\']\n    ].to_csv(dest/\'role_readiness.csv\',index=False,encoding=\'utf-8-sig\')\n    metadata={\'assessed_at\':str(as_of),\'release\':release.name,\'hash_checks\':len(manifest.get(\'files\',{})),\n              \'issues\':canonical_issues,\'missing_due_periods\':len(pending),\n              \'pending_publication_periods\':int((gap_df[\'狀態\']==\'PENDING_PUBLICATION\').sum()),\n              \'stage\':summary.to_dict(\'records\'),\'role\':roles.to_dict(\'records\')}\n    (dest/\'驗收摘要.json\').write_text(json.dumps(metadata,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf-8\')\n    return {\'summary\':summary,\'datasets\':datasets,\'roles\':roles,\'gaps\':gap_df,\n            \'recommendations\':recommendations,\'metadata\':metadata}\n\n\ndef show_zh_dashboard(result):\n    from IPython.display import display,Markdown\n    display(Markdown(\'## 資料收集與模型試訓｜繁體中文自動驗收總表\'))\n    display(Markdown(\'### ① 可否進入下一階段\'))\n    display(result[\'summary\'])\n    display(Markdown(\'### ② 資料已收集完成：各類型達標情況\'))\n    display(result[\'datasets\'])\n    display(Markdown(\'### ③ 可開始模型試訓：角色、可用期間與阻塞\'))\n    display(result[\'roles\'])\n    display(Markdown(\'### ④ 尚缺資料：精確月份、上市／上櫃與缺漏原因\'))\n    missing=result[\'gaps\'][result[\'gaps\'][\'狀態\'].isin([\'PARTIAL\',\'BLOCKED\'])]\n    display(missing if len(missing)<=70 else missing.head(70))\n    if len(missing)>70:\n        display(Markdown(f\'尚有 **{len(missing)}** 筆逐期缺口，請查看 Drive `reports/逐期缺口與原因.csv` 全表。\'))\n    wait=result[\'gaps\'][result[\'gaps\'][\'狀態\']==\'PENDING_PUBLICATION\']\n    if len(wait):display(Markdown(f\'另有 **{len(wait)}** 筆尚在公告期，不列為補收失敗。\'))\n    display(Markdown(\'### ⑤ 建議補收：只處理未達標的月份與資料集\'))\n    display(result[\'recommendations\'] if len(result[\'recommendations\'])<=70 else result[\'recommendations\'].head(70))\n    display(Markdown(\'**注意：PASS 僅為資料研究／試訓准入，不代表 OOS、PIT 或正式部署通過。**\'))\n', 'research_collection.py': '"""Janus four-role research collection. Standalone, no training or publication."""\nfrom __future__ import annotations\nimport calendar\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport re\nimport time\nfrom datetime import date, datetime, timezone\nfrom pathlib import Path\nimport pandas as pd\nimport requests\nfrom collect_tw_equities import execute as collect_ohlcv\nfrom monthly_folder_io import load_monthly_folders\n\nVERSION = \'collection-v1.0.0\'\nAPI = \'https://api.finmindtrade.com/api/v4/data\'\nDATASETS = {\n    \'revenue\': \'TaiwanStockMonthRevenue\',\n    \'valuation\': \'TaiwanStockPER\',\n    \'income\': \'TaiwanStockFinancialStatements\',\n    \'balance\': \'TaiwanStockBalanceSheet\',\n    \'cashflow\': \'TaiwanStockCashFlowsStatement\',\n    \'dividend\': \'TaiwanStockDividendResult\',\n    \'reduction\': \'TaiwanStockCapitalReductionReferencePrice\',\n    \'split\': \'TaiwanStockSplitPrice\',\n    \'par_value\': \'TaiwanStockParValueChange\',\n    \'industry\': \'TaiwanStockInfo\',\n    \'delisting\': \'TaiwanStockDelisting\',\n    \'benchmark_total_return\': \'TaiwanStockTotalReturnIndex\',\n}\nGLOBAL = {\'industry\', \'delisting\', \'split\', \'par_value\'}\nEVENTS = {\'dividend\', \'reduction\', \'split\', \'par_value\', \'delisting\'}\n\ndef now():\n    return datetime.now(timezone.utc).isoformat()\n\ndef sha(body):\n    return hashlib.sha256(body).hexdigest()\n\ndef encode(obj):\n    return json.dumps(obj, ensure_ascii=False, sort_keys=True, allow_nan=False).encode(\'utf-8\')\n\ndef atomic(path, body):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    with temp.open(\'wb\') as f:\n        f.write(body)\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(temp, path)\n\ndef month_range(start, end):\n    return [str(x) for x in pd.period_range(start, end, freq=\'M\')]\n\ndef clean_number(value):\n    if value is None or str(value).strip() in (\'\', \'--\', \'-\', \'N/A\', \'null\'):\n        return None\n    n = float(str(value).replace(\',\', \'\').strip())\n    if not math.isfinite(n):\n        raise ValueError(\'NONFINITE_NUMBER\')\n    return n\n\ndef normalise(kind, rows, task, roster, observed):\n    """Preserve provider period basis; never invent announcement dates or EPS."""\n    accepted, rejected, seen, keys = [], [], set(), set()\n    for original in rows:\n        try:\n            if not isinstance(original, dict):\n                raise ValueError(\'ROW_NOT_OBJECT\')\n            symbol = str(original.get(\'stock_id\', \'\')).strip()\n            if task[\'symbol\'] != \'*\' and symbol != task[\'symbol\']:\n                raise ValueError(\'WRONG_SYMBOL\')\n            if kind != \'benchmark_total_return\' and kind != \'delisting\' and symbol not in roster:\n                continue\n            if kind == \'industry\' and original.get(\'type\') != roster[symbol][\'market\']:\n                continue\n            raw_date = str(original.get(\'date\', \'\'))\n            if kind == \'industry\':\n                period = observed[:10]\n            elif kind == \'delisting\':\n                period = str(original.get(\'date\', original.get(\'delisting_date\', \'\')))\n                date.fromisoformat(period)\n            else:\n                period = date.fromisoformat(raw_date).isoformat()\n            if kind not in GLOBAL and not task[\'start\'] <= period <= task[\'end\']:\n                raise ValueError(\'OUTSIDE_REQUEST_RANGE\')\n            row = dict(original)\n            row.update(symbol=symbol, market=roster.get(symbol, {}).get(\'market\',\n                       {\'TAIEX\': \'twse\', \'TPEx\': \'tpex\'}.get(symbol, \'unknown\')),\n                       source_id=\'finmind\', dataset=DATASETS[kind], observed_at=observed,\n                       published_at=None, available_at=observed,\n                       historical_pit=\'NOT_VERIFIED\', provider_date=raw_date)\n            if kind == \'revenue\':\n                year, month = int(original[\'revenue_year\']), int(original[\'revenue_month\'])\n                fiscal = date(year, month, 1).strftime(\'%Y-%m\')\n                period = date(year, month, calendar.monthrange(year, month)[1]).isoformat()\n                if period >= raw_date:\n                    raise ValueError(\'INVALID_FISCAL_PERIOD\')\n                row.update(fiscal_month=fiscal, revenue_twd=clean_number(original[\'revenue\']),\n                           provider_observed_date=original.get(\'create_time\') or None)\n                if row[\'revenue_twd\'] is None or row[\'revenue_twd\'] < 0:\n                    raise ValueError(\'INVALID_REVENUE\')\n                key = (symbol, fiscal)\n            elif kind == \'valuation\':\n                for target, source in [(\'pe_ratio\', \'PER\'), (\'pb_ratio\', \'PBR\'),\n                                       (\'dividend_yield_percent\', \'dividend_yield\')]:\n                    if source not in original:\n                        raise ValueError(\'SCHEMA_\' + source)\n                    value = clean_number(original[source])\n                    row[target] = value if value is not None and value > 0 else None\n                key = (symbol, period)\n            elif kind in {\'income\', \'balance\', \'cashflow\'}:\n                if not original.get(\'type\') or \'value\' not in original:\n                    raise ValueError(\'FINANCIAL_SCHEMA\')\n                row.update(value=clean_number(original[\'value\']),\n                           fiscal_period_end=period, period_basis=\'PROVIDER_REPORTED_UNVERIFIED\',\n                           unit_basis=\'PROVIDER_REPORTED_UNVERIFIED\')\n                key = (symbol, period, str(original[\'type\']))\n            elif kind == \'benchmark_total_return\':\n                row[\'close\'] = clean_number(original[\'price\'])\n                if row[\'close\'] is None or row[\'close\'] <= 0:\n                    raise ValueError(\'INVALID_INDEX\')\n                row[\'return_basis\'] = \'TOTAL_RETURN_NOT_PRICE_INDEX\'\n                key = (symbol, period)\n            elif kind == \'industry\':\n                if not original.get(\'industry_category\'):\n                    raise ValueError(\'INDUSTRY_SCHEMA\')\n                row[\'classification_basis\'] = \'CURRENT_SNAPSHOT_NOT_HISTORICAL\'\n                key = (symbol, original.get(\'type\'))\n            else:\n                row[\'adjustment_status\'] = \'RAW_EVENT_NOT_VALIDATED_ADJUSTMENT\'\n                key = (symbol, period, str(original.get(\'type\', kind)))\n            fingerprint = sha(encode(original))\n            identity = (key, fingerprint)\n            if identity in seen:\n                continue\n            if key in keys:\n                raise ValueError(\'CONFLICTING_DUPLICATE\')\n            seen.add(identity)\n            keys.add(key)\n            row[\'partition_month\'] = period[:7]\n            accepted.append(row)\n        except (ValueError, KeyError, TypeError, OverflowError) as exc:\n            rejected.append({\'row\': original, \'reason\': str(exc)[:100]})\n    return accepted, rejected\n\ndef audit_ohlcv(root, start, end):\n    reports, roster = [], {}\n    for month in month_range(start, end):\n        folder = Path(root) / \'data\' / month\n        if not list(folder.glob(\'manifest-r[0-9]*.json\')):\n            reports.append({\'month\': month, \'status\': \'MISSING_OR_CHECKPOINT\', \'rows\': 0})\n            continue\n        try:\n            # Existing canonical research loader verifies all member hashes and market coverage.\n            df, provenance = load_monthly_folders([folder])\n            manifests = list(folder.glob(\'manifest-r[0-9]*.json\'))\n            active = max(manifests, key=lambda p:int(re.fullmatch(r\'manifest-r(\\d+)\\.json\',p.name).group(1)))\n            revision = json.loads(active.read_text(\'utf-8\'))[\'revision\']\n            universe = pd.read_csv(folder / f\'universe-{revision}.csv\', dtype=str)\n            # Newer roster replaces stale r1 roster; old months stay immutable.\n            roster = {r[\'symbol\']:r for r in universe.to_dict(\'records\')}\n            bad = df[\'close\'].isna() | (df[\'close\'] <= 0) | df[\'volume\'].isna() | (df[\'volume\'] < 0)\n            for col in (\'open\', \'high\', \'low\'):\n                if col in df:\n                    bad |= df[col].notna() & (df[col] <= 0)\n            if {\'open\', \'high\', \'low\'}.issubset(df):\n                bad |= (df[\'high\'] < df[[\'open\', \'close\', \'low\']].max(axis=1))\n                bad |= (df[\'low\'] > df[[\'open\', \'close\', \'high\']].min(axis=1))\n            for col in (\'open\', \'high\', \'low\', \'close\', \'volume\', \'turnover\'):\n                if col in df:\n                    bad |= df[col].notna() & ~df[col].map(lambda x: pd.isna(x) or math.isfinite(float(x)))\n            if \'turnover\' in df:\n                bad |= df[\'turnover\'].notna() & (df[\'turnover\'] < 0)\n            clean_dir = Path(root) / \'_derived\' / \'ohlcv-clean-v1\' / month / provenance[0][\'sha256\'][:16]\n            output_hashes = {}\n            for filename, frame in [(\'clean.csv.gz\', df.loc[~bad]), (\'rejected.csv.gz\', df.loc[bad])]:\n                content = gzip.compress(frame.to_csv(index=False).encode(\'utf-8\'), mtime=0)\n                dest = clean_dir / filename\n                if dest.exists() and dest.read_bytes() != content:\n                    raise ValueError(\'clean output changed; upgrade cleaner version\')\n                if not dest.exists():\n                    atomic(dest, content)\n                output_hashes[filename] = sha(content)\n            if not (clean_dir / \'manifest.json\').exists():\n                atomic(clean_dir / \'manifest.json\', encode({\'source_manifest_sha256\': provenance[0][\'sha256\'],\n                    \'sha256\': output_hashes, \'adjustment_basis\':\'UNADJUSTED\', \'publication_allowed\':False}))\n            reports.append({\'month\': month, \'status\': \'INVALID_ROWS\' if bad.any() else \'VERIFIED_RESEARCH\',\n                            \'rows\': len(df), \'invalid_rows\': int(bad.sum()),\n                            \'missing_ohlc_rows\': int(df[[\'open\', \'high\', \'low\']].isna().any(axis=1).sum()),\n                            \'markets\': provenance[0][\'research_coverage\'][\'markets\'],\n                            \'manifest_sha256\': provenance[0][\'sha256\']})\n        except (ValueError, KeyError, OSError) as exc:\n            reports.append({\'month\': month, \'status\': \'FAILED_VALIDATION\', \'reason\': str(exc)[:160], \'rows\': 0})\n    return reports, roster\n\ndef build_tasks(roster, end_month, selected=None):\n    end = str(pd.Period(end_month, \'M\').end_time.date())\n    starts = {\'revenue\': str(pd.Period(end_month, \'M\') - 47) + \'-01\',\n              \'valuation\': str(pd.Period(end_month, \'M\') - 35) + \'-01\',\n              \'income\': str(pd.Period(end_month, \'M\') - 47) + \'-01\',\n              \'balance\': str(pd.Period(end_month, \'M\') - 47) + \'-01\',\n              \'cashflow\': str(pd.Period(end_month, \'M\') - 47) + \'-01\'}\n    symbols = sorted(roster)\n    if selected:\n        unknown = set(selected) - set(symbols)\n        if unknown:\n            raise ValueError(\'股票不在普通股名冊: \' + \',\'.join(sorted(unknown)))\n        symbols = sorted(set(selected))\n    tasks = []\n    for kind in [\'industry\', \'delisting\', \'split\', \'par_value\', \'benchmark_total_return\']:\n        for symbol in ([\'TAIEX\', \'TPEx\'] if kind == \'benchmark_total_return\' else [\'*\']):\n            tasks.append(dict(kind=kind, symbol=symbol, start=\'2025-01-01\', end=end))\n    # Stock-major order yields complete research bundles early; each request covers full history.\n    for symbol in symbols:\n        for kind in [\'revenue\', \'valuation\', \'income\', \'balance\', \'cashflow\', \'dividend\', \'reduction\']:\n            tasks.append(dict(kind=kind, symbol=symbol, start=starts.get(kind, \'2025-01-01\'), end=end))\n    for t in tasks:\n        t[\'id\'] = sha(encode(t))[:24]\n    return tasks\n\nclass Collector:\n    def __init__(self, root, tasks, roster, *, max_requests=100, max_minutes=30,\n                 interval=15, token=\'\', session=None, clock=time.monotonic, sleep=time.sleep):\n        if not (1 <= max_requests <= 500 and 2 <= max_minutes <= 60 and interval >= 15):\n            raise ValueError(\'請求1–500、時間2–60分鐘、間隔至少15秒\')\n        self.root, self.tasks, self.roster = Path(root), tasks, roster\n        self.session = session or requests.Session()\n        self.clock, self.sleep, self.interval = clock, sleep, interval\n        self.deadline = clock() + max_minutes * 60\n        self.remaining, self.last = max_requests, None\n        self.headers = {\'User-Agent\': \'JanusResearch/1.0\'}\n        if token:\n            self.headers[\'Authorization\'] = \'Bearer \' + token\n        self.plan_hash = sha(encode({\'version\': VERSION, \'tasks\': tasks, \'roster\': roster}))\n        self.base = self.root / \'_collection\' / self.plan_hash[:16]\n        self.base.mkdir(parents=True, exist_ok=True)\n        plan = self.base / \'plan.json\'\n        expected = encode({\'version\': VERSION, \'hash\': self.plan_hash, \'tasks\': tasks, \'roster\': roster})\n        if plan.exists() and plan.read_bytes() != expected:\n            raise ValueError(\'PLAN_COLLISION\')\n        if not plan.exists():\n            atomic(plan, expected)\n\n    def chunks(self):\n        for p in sorted(self.base.glob(\'batch-*.json.gz\')):\n            block = json.loads(gzip.decompress(p.read_bytes()))\n            if block[\'plan_hash\'] != self.plan_hash:\n                raise ValueError(\'CHECKPOINT_PLAN_MISMATCH\')\n            for rec in block[\'records\'].values():\n                if sha(encode(rec[\'raw_rows\'])) != rec[\'raw_rows_sha256\']:\n                    raise ValueError(\'CHECKPOINT_HASH_MISMATCH\')\n            yield p, block\n\n    def records(self):\n        return {k: v for _, block in self.chunks() for k, v in block[\'records\'].items()}\n\n    def save(self, index, rec):\n        p = self.base / f\'batch-{index // 20:05d}.json.gz\'\n        block = json.loads(gzip.decompress(p.read_bytes())) if p.exists() else {\'plan_hash\': self.plan_hash, \'records\': {}}\n        block[\'records\'][self.tasks[index][\'id\']] = rec\n        atomic(p, gzip.compress(encode(block), mtime=0))\n\n    def request(self, task):\n        params = {\'dataset\': DATASETS[task[\'kind\']]}\n        if task[\'kind\'] not in GLOBAL:\n            params.update(data_id=task[\'symbol\'], start_date=task[\'start\'], end_date=task[\'end\'])\n        for attempt in range(3):\n            if self.remaining <= 0 or self.clock() >= self.deadline - 45:\n                raise RuntimeError(\'TIME_OR_REQUEST_LIMIT\')\n            if self.last is not None:\n                delay = max(0, self.interval - (self.clock() - self.last))\n                if self.clock() + delay >= self.deadline - 45:\n                    raise RuntimeError(\'TIME_OR_REQUEST_LIMIT\')\n                self.sleep(delay)\n            self.last = self.clock()\n            self.remaining -= 1\n            try:\n                response = self.session.get(API, params=params, headers=self.headers, timeout=30)\n                if response.status_code in (401, 402, 403, 429):\n                    raise RuntimeError(\'AUTH_OR_QUOTA_\' + str(response.status_code))\n                if response.status_code >= 500:\n                    raise requests.ConnectionError(\'SERVER_UNAVAILABLE\')\n                if response.status_code != 200:\n                    raise RuntimeError(\'HTTP_\' + str(response.status_code))\n                obj = response.json()\n                if obj.get(\'status\') != 200:\n                    # No raw error bodies/headers: provider messages can contain credentials.\n                    raise RuntimeError(\'PROVIDER_STATUS_\' + str(obj.get(\'status\')))\n                rows = obj.get(\'data\')\n                if not isinstance(rows, list) or len(rows) > 200000:\n                    raise RuntimeError(\'INVALID_RESPONSE_SCHEMA_OR_SIZE\')\n                return rows, sha(response.content)\n            except (requests.RequestException, ValueError):\n                if attempt == 2:\n                    raise RuntimeError(\'NETWORK_OR_JSON_ERROR\') from None\n        raise AssertionError(\'unreachable\')\n\n    def run(self, retry_empty=False):\n        existing = self.records()\n        failures, done, stop = [], 0, \'COMPLETE_REQUEST_QUEUE\'\n        for index, task in enumerate(self.tasks):\n            old = existing.get(task[\'id\'])\n            if old and old[\'status\'] != \'SCHEMA_REJECTED\' and not (retry_empty and old[\'status\'] == \'EMPTY_RESPONSE\'):\n                continue\n            try:\n                raw, response_hash = self.request(task)\n                observed = now()\n                rows, rejects = normalise(task[\'kind\'], raw, task, self.roster, observed)\n                # Raw succeeds but parser failure never masquerades as finished collection.\n                status = (\'EMPTY_RESPONSE\' if not raw else \'SCHEMA_REJECTED\' if rejects else\n                          \'NO_ELIGIBLE_ROWS\' if not rows else \'COLLECTED\')\n                rec = dict(task=task, status=status, observed_at=observed, raw_rows=raw,\n                           raw_rows_sha256=sha(encode(raw)), response_sha256=response_hash,\n                           rows=rows, rejected=rejects, source_url=API, publication_allowed=False)\n                self.save(index, rec)\n                done += 1\n                print(f"{index+1}/{len(self.tasks)} {task[\'kind\']} {task[\'symbol\']}: {status}, {len(rows)} rows")\n            except RuntimeError as exc:\n                code = str(exc)\n                failures.append(dict(task=task, error=code, at=now()))\n                if code == \'TIME_OR_REQUEST_LIMIT\' or code.startswith((\'AUTH_OR_QUOTA\', \'PROVIDER_STATUS\')):\n                    stop = code\n                    break\n                if len(failures) >= 3:\n                    stop = \'THREE_FAILURES_STOP\'\n                    break\n        summary = {\'stop_reason\': stop, \'new_requests_saved\': done, \'failures\': failures,\n                   \'at\': now(), \'publication_allowed\': False}\n        atomic(self.base / \'last_run.json\', encode(summary))\n        return summary\n\n    def export(self):\n        """Immutable generation; month shards committed by manifest-last, resumable from raw."""\n        records = self.records()\n        generation = sha(encode({k: (r[\'raw_rows_sha256\'], r[\'observed_at\']) for k, r in records.items()}))[:16]\n        dest = self.base / \'exports\' / generation\n        if (dest / \'manifest.json\').exists():\n            manifest = json.loads((dest / \'manifest.json\').read_text(\'utf-8\'))\n            for name, expected in manifest[\'sha256\'].items():\n                if sha((dest / name).read_bytes()) != expected:\n                    raise ValueError(\'EXPORT_HASH_MISMATCH\')\n            return dest\n        partitions = {}\n        for rec in records.values():\n            for row in rec[\'rows\']:\n                key = (row[\'partition_month\'], rec[\'task\'][\'kind\'])\n                partitions.setdefault(key, []).append(row)\n        hashes = {}\n        for (month, kind), rows in sorted(partitions.items()):\n            name = f\'{month}/{kind}.csv.gz\'\n            body = gzip.compress(pd.DataFrame(rows).to_csv(index=False).encode(\'utf-8\'), mtime=0)\n            path = dest / name\n            if path.exists() and path.read_bytes() != body:\n                raise ValueError(\'IMMUTABLE_EXPORT_CONFLICT\')\n            if not path.exists():\n                atomic(path, body)\n            hashes[name] = sha(body)\n        manifest = dict(version=VERSION, plan_hash=self.plan_hash, generation=generation,\n                        sha256=hashes, created_at=now(), publication_allowed=False,\n                        historical_pit=\'NOT_VERIFIED\', corporate_adjustment=\'NOT_APPLIED\',\n                        source=\'finmind\', baseline_ohlcv=\'original r1 remains unchanged\')\n        atomic(dest / \'manifest.json\', encode(manifest))\n        return dest\n\n    def report(self, ohlcv):\n        records = self.records()\n        summary = []\n        gaps = []\n        for kind in DATASETS:\n            tasks = [t for t in self.tasks if t[\'kind\'] == kind]\n            found = [records[t[\'id\']] for t in tasks if t[\'id\'] in records]\n            good = [r for r in found if r[\'status\'] == \'COLLECTED\']\n            rows = [row for r in good for row in r[\'rows\']]\n            summary.append(dict(dataset=kind, requests_planned=len(tasks), responses_saved=len(found),\n                                requests_pending=len(tasks)-len(found), successful_nonempty=len(good),\n                                empty_responses=sum(r[\'status\']==\'EMPTY_RESPONSE\' for r in found),\n                                schema_rejected=sum(r[\'status\']==\'SCHEMA_REJECTED\' for r in found),\n                                rows=len(rows), independent_months=len({r[\'partition_month\'] for r in rows}),\n                                symbols=len({r[\'symbol\'] for r in rows})))\n            for t in tasks:\n                rec = records.get(t[\'id\'])\n                if kind in {\'revenue\',\'valuation\',\'income\',\'balance\',\'cashflow\',\'benchmark_total_return\'}:\n                    freq = \'Q\' if kind in {\'income\',\'balance\',\'cashflow\'} else \'M\'\n                    expected = {str(x) for x in pd.period_range(t[\'start\'],t[\'end\'],freq=freq)}\n                    actual = {str(pd.Period(r[\'partition_month\'],freq=freq)) for r in (rec or {}).get(\'rows\',[])}\n                    missing = sorted(expected-actual)\n                    gaps.append(dict(dataset=kind,symbol=t[\'symbol\'],expected_native_periods=len(expected),\n                                     observed_native_periods=len(actual & expected),missing_periods=\',\'.join(missing),\n                                     request_status=(rec or {}).get(\'status\',\'PENDING\'),\n                                     note=\'依要求窗口列缺期；IPO前或尚未公告期不自動認定為來源錯誤\'))\n        per_month = []\n        for kind in DATASETS:\n            relevant = [r for r in records.values() if r[\'task\'][\'kind\'] == kind and r[\'status\']==\'COLLECTED\']\n            rows = [r for rec in relevant for r in rec[\'rows\']]\n            if not rows:\n                continue\n            df = pd.DataFrame(rows)\n            for (month, market), group in df.groupby([\'partition_month\', \'market\']):\n                denominator = sum(r[\'market\']==market for r in self.roster.values())\n                if kind == \'benchmark_total_return\':\n                    denominator = 1\n                ratio = group.symbol.nunique()/denominator if denominator else None\n                per_month.append(dict(dataset=kind, month=month, market=market, rows=len(group),\n                                      symbols=group.symbol.nunique(), denominator=denominator,\n                                      coverage=ratio, basis=\'CURRENT_ROSTER_PROBE\',\n                                      threshold_applies=kind not in EVENTS and kind!=\'industry\'))\n        availability = {x[\'dataset\']: x[\'successful_nonempty\'] > 0 for x in summary}\n        dependencies = {\'Quant\':[\'benchmark_total_return\',\'dividend\',\'reduction\'],\n                        \'Risk\':[\'dividend\',\'reduction\'], \'Fundamental\':[\'revenue\'],\n                        \'Valuation\':[\'valuation\',\'income\',\'balance\',\'industry\']}\n        roles = []\n        for role, needs in dependencies.items():\n            missing = [x for x in needs if not availability[x]]\n            roles.append(dict(role=role, collection_status=\'PARTIAL\' if missing else \'DATA_PRESENT_CHECK_COVERAGE\',\n                              missing_datasets=\',\'.join(missing), training_ready=False,\n                              remaining_checks={\'Quant\':\'公司行動調整、價格/含息基準一致、交易日、成熟標籤\',\n                              \'Risk\':\'公司行動調整、交易日、成熟標籤；既有未調整PROBE可沿用\',\n                              \'Fundamental\':\'逐股連續月份、公告時點、首次版/修訂；不把provider date當公告\',\n                              \'Valuation\':\'財報單位/單季口徑、as-of、有效同業、歷史產業\'}[role]))\n        root = self.base / \'reports\'\n        for filename, data in [(\'ohlcv_audit\',ohlcv), (\'collection_summary\',summary),\n                               (\'monthly_coverage\',per_month), (\'native_period_gaps\',gaps), (\'role_readiness\',roles)]:\n            atomic(root / (filename+\'.csv\'), pd.DataFrame(data).to_csv(index=False).encode(\'utf-8-sig\'))\n        atomic(root/\'report.json\', encode(dict(at=now(), ohlcv=ohlcv, datasets=summary,\n                                               monthly_coverage=per_month, roles=roles,\n                                               publication_allowed=False)))\n        return pd.DataFrame(summary), pd.DataFrame(roles), root\n', 'canonical_v3.py': '"""Janus research dataset v3: a single read contract for CSV/JSON source archives.\n\nThis file is embedded in the Colab notebook. It does not fetch data or publish\ncanonical/production facts. The single model-facing format is partitioned Parquet.\n"""\nfrom __future__ import annotations\nimport gzip\nimport hashlib\nimport json\nimport re\nimport shutil\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport pandas as pd\n\nVERSION = \'3.1.0\'\nOFFICIAL = {\'twse\', \'tpex\', \'mops\', \'TWSE:MI_INDEX\', \'TPEX:stk_wn1430\'}\nFINANCE_KINDS = {\'income\', \'balance\', \'cashflow\'}\nEVENT_KINDS = {\'dividend\', \'reduction\', \'split\', \'par_value\', \'delisting\', \'industry\',\n               \'benchmark_total_return\'}\nSCHEMAS = {\n    \'ohlcv\': [\'symbol\',\'market\',\'trade_date\',\'open\',\'high\',\'low\',\'close\',\'volume\',\n              \'turnover\',\'source_id\',\'observed_at\'],\n    \'revenue\': [\'symbol\',\'market\',\'fiscal_month\',\'revenue_twd\',\'source_id\',\n                \'unit_basis\',\'observed_at\',\'published_at\',\'historical_pit\'],\n    \'valuation\': [\'symbol\',\'market\',\'trade_date\',\'pe_ratio\',\'pb_ratio\',\n                  \'dividend_yield_percent\',\'source_id\',\'observed_at\',\'published_at\',\n                  \'historical_pit\'],\n    \'financials\': [\'symbol\',\'market\',\'dataset\',\'fiscal_quarter\',\'item\',\'value\',\n                   \'unit_basis\',\'period_basis\',\'source_id\',\'observed_at\',\n                   \'published_at\',\'historical_pit\'],\n    \'corporate_actions\': [\'symbol\',\'market\',\'dataset\',\'trade_date\',\'fiscal_month\',\n                          \'fiscal_quarter\',\'event_payload_json\',\'source_id\',\n                          \'observed_at\',\'published_at\',\'historical_pit\'],\n}\nKEYS = {\n    \'ohlcv\': [\'symbol\',\'market\',\'trade_date\'],\n    \'revenue\': [\'symbol\',\'market\',\'fiscal_month\'],\n    \'valuation\': [\'symbol\',\'market\',\'trade_date\'],\n    \'financials\': [\'symbol\',\'market\',\'dataset\',\'fiscal_quarter\',\'item\',\'unit_basis\'],\n    \'corporate_actions\': [\'symbol\',\'market\',\'dataset\',\'trade_date\',\'fiscal_month\',\n                          \'fiscal_quarter\',\'event_payload_json\'],\n}\n\n\ndef _json(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str,\n                      allow_nan=False, separators=(\',\',\':\'))\n\n\ndef _digest(body: bytes) -> str:\n    return hashlib.sha256(body).hexdigest()\n\n\ndef _kind(record: dict) -> str | None:\n    kind = record.get(\'dataset\')\n    if kind in (\'ohlcv\', \'revenue\', \'valuation\'):\n        return kind\n    if kind in FINANCE_KINDS:\n        return \'financials\'\n    if kind in EVENT_KINDS:\n        return \'corporate_actions\'\n    return None\n\n\ndef _partition(table: str, row: dict) -> str:\n    if table in (\'ohlcv\', \'valuation\'):\n        value = str(row.get(\'trade_date\') or \'\')[:7]\n    elif table == \'revenue\':\n        value = str(row.get(\'fiscal_month\') or \'\')[:7]\n    elif table == \'financials\':\n        quarter = str(row.get(\'fiscal_quarter\') or \'\')\n        match = re.fullmatch(r\'(\\d{4})Q([1-4])\', quarter)\n        value = f\'{match[1]}-{int(match[2])*3:02d}\' if match else \'\'\n    else:\n        value = str(row.get(\'partition_month\') or row.get(\'trade_date\') or \'\')[:7]\n    if not re.fullmatch(r\'20\\d{2}-(0[1-9]|1[0-2])\', value):\n        raise ValueError(f\'INVALID_PARTITION_{table}: {value!r}\')\n    return value\n\n\ndef _score(record: dict) -> int:\n    source = str(record.get(\'source_id\', \'\')).lower()\n    if source in (\'twse\',\'tpex\',\'mops\') or source.startswith((\'twse:\', \'tpex:\')):\n        return 3\n    if source == \'finmind\':\n        return 2\n    if source == \'yahoo\':\n        return 1\n    return 0\n\n\ndef _standardize(table: str, record: dict) -> dict:\n    out = {k: record.get(k) for k in SCHEMAS[table]}\n    out[\'symbol\'] = str(out[\'symbol\']).strip()\n    out[\'market\'] = str(out[\'market\']).strip().lower()\n    if not out[\'symbol\'] or out[\'symbol\'] == \'None\' or out[\'market\'] not in (\'twse\',\'tpex\'):\n        raise ValueError(\'INVALID_SYMBOL_OR_MARKET\')\n    out[\'source_id\'] = str(record.get(\'source_id\') or record.get(\'source\') or \'unknown\')\n    if table == \'ohlcv\':\n        for col in (\'open\',\'high\',\'low\',\'close\',\'volume\',\'turnover\'):\n            out[col] = pd.to_numeric(out[col], errors=\'coerce\')\n        if any(pd.isna(out[c]) for c in (\'open\',\'high\',\'low\',\'close\')):\n            raise ValueError(\'OHLCV_REQUIRED_PRICE_MISSING\')\n        if out[\'high\'] < out[\'low\'] or out[\'close\'] <= 0 or out[\'volume\'] < 0:\n            raise ValueError(\'OHLCV_INVALID_PRICE\')\n    elif table == \'financials\':\n        out[\'dataset\'] = record[\'dataset\']\n        out[\'item\'] = str(record.get(\'item\') or record.get(\'type\') or \'\')\n        if not out.get(\'fiscal_quarter\'):\n            provider_end = str(record.get(\'fiscal_period_end\') or record.get(\'provider_date\') or \'\')\n            hit = re.fullmatch(r\'(20\\d{2})-(03|06|09|12)-(\\d{2})\', provider_end)\n            if hit:\n                out[\'fiscal_quarter\'] = f"{hit[1]}Q{int(hit[2])//3}"\n                out[\'period_basis\'] = \'PROVIDER_QUARTER_END_INFERRED\'\n        if not out.get(\'fiscal_quarter\'):\n            raise ValueError(\'FINANCIAL_QUARTER_NOT_MAPPABLE\')\n        out[\'unit_basis\'] = str(record.get(\'unit_basis\') or \'UNKNOWN\')\n        if not out[\'item\'] or pd.isna(pd.to_numeric(out[\'value\'], errors=\'coerce\')):\n            raise ValueError(\'FINANCIAL_ITEM_OR_VALUE_MISSING\')\n        out[\'value\'] = float(out[\'value\'])\n    elif table == \'corporate_actions\':\n        out[\'dataset\'] = record[\'dataset\']\n        payload = {k: v for k, v in record.items() if k not in SCHEMAS[table]\n                   and k not in (\'partition_month\',\'publication_allowed\')}\n        out[\'event_payload_json\'] = _json(payload)\n    elif table == \'revenue\':\n        out[\'revenue_twd\'] = pd.to_numeric(out[\'revenue_twd\'], errors=\'coerce\')\n        if pd.isna(out[\'revenue_twd\']) or out[\'revenue_twd\'] < 0:\n            raise ValueError(\'REVENUE_MISSING_OR_NEGATIVE\')\n    elif table == \'valuation\':\n        if not out.get(\'trade_date\'):\n            out[\'trade_date\'] = record.get(\'provider_date\')\n        if all(pd.isna(pd.to_numeric(out[c],errors=\'coerce\')) for c in\n               (\'pe_ratio\',\'pb_ratio\',\'dividend_yield_percent\')):\n            raise ValueError(\'VALUATION_ALL_FIELDS_EMPTY\')\n        for c in (\'pe_ratio\',\'pb_ratio\',\'dividend_yield_percent\'):\n            out[c] = pd.to_numeric(out[c], errors=\'coerce\')\n            if pd.isna(out[c]):\n                out[c] = None\n    _partition(table, {**record, **out})\n    return out\n\n\ndef _load_yahoo(root: Path) -> list[dict]:\n    rows = []\n    for path in sorted((root/\'data\').glob(\'20??-??/yahoo-*.json.gz\')):\n        records = json.loads(gzip.decompress(path.read_bytes()))\n        if not isinstance(records, list):\n            raise ValueError(f\'YAHOO_FORMAT_MISMATCH: {path}\')\n        rows.extend(records)\n    return rows\n\n\ndef _load_ohlcv(root: Path) -> list[dict]:\n    rows = []\n    for folder in sorted((root/\'data\').glob(\'20??-??\')):\n        manifest_files=list(folder.glob(\'manifest-r[0-9]*.json\'))\n        if not manifest_files: continue\n        chosen=max(manifest_files, key=lambda p:int(re.fullmatch(r\'manifest-r(\\d+)\\.json\',p.name).group(1)))\n        revision=json.loads(chosen.read_text(\'utf-8\'))[\'revision\']\n        path=folder/f\'ohlcv-{revision}.csv.gz\'\n        if not path.exists(): raise ValueError(f\'OHLCV_MISSING: {folder.name} {revision}\')\n        frame = pd.read_csv(path, compression=\'gzip\', dtype={\'symbol\':str,\'market\':str})\n        if \'source\' in frame and \'source_id\' not in frame:\n            frame = frame.rename(columns={\'source\':\'source_id\'})\n        if frame.empty:\n            continue\n        if not frame[\'trade_date\'].astype(str).str.startswith(path.parent.name+\'-\').all():\n            raise ValueError(f\'OHLCV_MONTH_MISMATCH: {path}\')\n        for rec in frame.to_dict(\'records\'):\n            rec[\'dataset\'] = \'ohlcv\'\n            rows.append(rec)\n    return rows\n\n\ndef choose_records(records: list[dict]):\n    """Stable source-priority selection; keep a conflict receipt for disagreement.\n\n    This normalizes field names and storage type, NOT fundamental metric semantics.\n    For different financial item/units, keep independent rows.\n    """\n    by_key, invalid, conflicts = {}, [], []\n    count = 0\n    for row in records:\n        table = _kind(row)\n        if table is None:\n            invalid.append({\'reason\':\'UNSUPPORTED_DATASET\',\'row\':str(row)[:400]})\n            continue\n        try:\n            clean = _standardize(table, row)\n            month = _partition(table, {**row, **clean})\n            key = (table, month, *(str(clean.get(k) or \'\') for k in KEYS[table]))\n            rank = _score(clean)\n            serialized = _json({k:clean[k] for k in SCHEMAS[table]\n                                if k not in (\'observed_at\',\'published_at\',\'historical_pit\')})\n            values_only = _json({k:clean[k] for k in SCHEMAS[table]\n                                 if k not in (\'source_id\',\'observed_at\',\'published_at\',\'historical_pit\')})\n            entry = (rank, serialized, clean, values_only)\n            existing = by_key.get(key)\n            if existing is not None:\n                if existing[3] != values_only:\n                    conflicts.append({\'dataset\':table,\'month\':month,\'key\':repr(key[2:]),\n                                      \'kept_source\':existing[2][\'source_id\'] if existing[0]>=rank else clean[\'source_id\'],\n                                      \'other_source\':clean[\'source_id\'] if existing[0]>=rank else existing[2][\'source_id\']})\n                if (rank,serialized) <= (existing[0],existing[1]):\n                    continue\n            by_key[key] = entry\n            count += 1\n        except (ValueError, TypeError, OverflowError) as exc:\n            invalid.append({\'reason\':str(exc)[:200],\'row\':str(row)[:400]})\n    partitions = {}\n    for key, (_, _, row, _) in sorted(by_key.items()):\n        partitions.setdefault((key[0],key[1]), []).append(row)\n    return partitions, invalid, conflicts\n\n\ndef build_canonical(root, supplemental_rows, *, run_tag=\'colab\', read_ohlcv=True):\n    """Create model-facing Parquet release and CURRENT pointer, with checksums.\n\n    Successful output does not assert PIT accuracy or permission to publish facts.\n    If pyarrow is unavailable, fails before CURRENT pointer replacement.\n    """\n    root = Path(root)\n    try:\n        import pyarrow  # noqa: F401\n    except ImportError as exc:\n        raise RuntimeError(\'請先安裝 pyarrow（Colab：pip install pyarrow）\') from exc\n    inputs = list(supplemental_rows) + _load_yahoo(root)\n    if read_ohlcv:\n        inputs.extend(_load_ohlcv(root))\n    parts, invalid, conflicts = choose_records(inputs)\n    if read_ohlcv and not any(k[0]==\'ohlcv\' for k in parts):\n        raise RuntimeError(\'OHLCV_MISSING: 不發布空白模型資料包\')\n    base = root/\'canonical\'\n    staging = base/\'_building\'\n    if staging.exists():\n        shutil.rmtree(staging)\n    staging.mkdir(parents=True, exist_ok=True)\n    manifest = {\'version\':VERSION,\'run_tag\':run_tag,\n                \'status\':\'RESEARCH_DATA_ONLY\',\'historical_pit\':\'NOT_VERIFIED\',\n                \'publication_allowed\':False,\'format\':\'parquet\',\n                \'tables\':{},\'files\':{},\'unresolved_conflict_rows\':len(conflicts),\n                \'invalid_rows\':len(invalid)}\n    try:\n        for (table,month),values in sorted(parts.items()):\n            out = staging/table/(month+\'.parquet\')\n            out.parent.mkdir(parents=True,exist_ok=True)\n            df = pd.DataFrame.from_records(values,columns=SCHEMAS[table])\n            df = df.sort_values(KEYS[table], kind=\'mergesort\').reset_index(drop=True)\n            df.to_parquet(out,index=False,engine=\'pyarrow\',compression=\'zstd\')\n            name=out.relative_to(staging).as_posix()\n            manifest[\'files\'][name]={\'sha256\':_digest(out.read_bytes()),\'rows\':len(df)}\n            manifest[\'tables\'][table]=manifest[\'tables\'].get(table,0)+len(df)\n        for name,rows in [(\'conflicts.csv\',conflicts),(\'invalid.csv\',invalid)]:\n            target=staging/name\n            pd.DataFrame(rows).to_csv(target,index=False,encoding=\'utf-8-sig\')\n            manifest[\'files\'][name]={\'sha256\':_digest(target.read_bytes()),\'rows\':len(rows)}\n        stable = {k:v for k,v in manifest.items() if k != \'run_tag\'}\n        release_id = _digest(_json(stable).encode(\'utf-8\'))[:16]\n        manifest[\'release_id\'] = release_id\n        (staging/\'manifest.json\').write_text(_json(manifest)+\'\\n\',encoding=\'utf-8\')\n        release = base/(\'release-\'+release_id)\n        if release.exists():\n            for name,expected in manifest[\'files\'].items():\n                if _digest((release/name).read_bytes()) != expected[\'sha256\']:\n                    raise ValueError(\'EXISTING_CANONICAL_DIGEST_MISMATCH\')\n            shutil.rmtree(staging)\n        else:\n            staging.rename(release)\n        pointer = base/\'CURRENT.json\'\n        temp=base/\'CURRENT.json.tmp\'\n        temp.write_text(_json({\'release\':release.name,\'manifest\':str(release.relative_to(root)/\'manifest.json\'),\n                               \'format\':\'parquet\',\'status\':\'RESEARCH_DATA_ONLY\'})+\'\\n\',encoding=\'utf-8\')\n        temp.replace(pointer)\n    except Exception:\n        if staging.exists():\n            shutil.rmtree(staging)\n        raise\n    return release, manifest\n', 'collect_tw_equities.py': '"""Janus D07-PRE research-only TWSE/TPEx common-stock historical daily collector.\n\nRun manually in Google Colab. Uses official day-scoped batch endpoints, one\nrequest per market/day (not one per stock/month), with immutable month folders.\nCompany master is *current* only; historical survivorship is NOT certified.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport os\nimport re\nimport time\nfrom datetime import date, timedelta, datetime, timezone\nfrom zoneinfo import ZoneInfo\nfrom pathlib import Path\nfrom typing import Any\n\nimport requests\n\nTWSE_COMPANIES = \'https://openapi.twse.com.tw/v1/opendata/t187ap03_L\'\nTPEX_COMPANIES = \'https://www.tpex.org.tw/openapi/v1/mopsfin_t187ap03_O\'\nTPEX_COMPANIES_CSV = \'https://mopsfin.twse.com.tw/opendata/t187ap03_O.csv\'\nTWSE_DAILY = \'https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX\'\nTPEX_DAILY = \'https://www.tpex.org.tw/web/stock/aftertrading/otc_quotes_no1430/stk_wn1430_result.php\'\nHEADERS = (\'symbol\', \'market\', \'trade_date\', \'open\', \'high\', \'low\', \'close\', \'volume\', \'turnover\', \'source\', \'observed_at\')\nEMPTY_TOKENS = {\'\', \'--\', \'---\', \'-\', \'－\', \'N/A\', \'null\', \'除權\', \'除息\'}\nEXCLUDE_NAME = re.compile(r\'ETF|ETN|指數投資|認購|認售|權證|存託憑證|特別股|受益憑證|債券|基金\', re.I)\n\n\n# Official exchange non-trading dates, limited to the approved 2025-2026 backfill.\n# 2025 TWSE holiday announcement and year schedule:\n#   https://eshop.twse.com.tw/zh/news/detail/8a82e9e69471d3e8019495d93c4f0019\n#   https://www.twse.com.tw/zh/trading/holiday.html\n# 2026 TWSE published calendar:\n#   https://openapi.twse.com.tw/v1/holidaySchedule/holidaySchedule\n# Scheduled "settlement only" sessions are CLOSED to securities trading.\n# These dates apply to TWSE + TPEx cash equities, NOT futures trading.\n# Unscheduled typhoon closures are not inferred or silently ignored.\n_SCHEDULED_CLOSURES = {\n    2025: (\n        \'01-01\', \'01-23\', \'01-24\', \'01-27\', \'01-28\', \'01-29\', \'01-30\', \'01-31\',\n        \'02-28\', \'04-03\', \'04-04\', \'05-01\', \'05-30\', \'09-29\', \'10-06\',\n        \'10-10\', \'10-24\', \'12-25\',\n    ),\n    2026: (\n        \'01-01\', \'02-12\', \'02-13\', \'02-16\', \'02-17\', \'02-18\', \'02-19\',\n        \'02-20\', \'02-27\', \'04-03\', \'04-06\', \'05-01\', \'06-19\', \'09-25\',\n        \'09-28\', \'10-09\', \'10-26\', \'12-25\',\n    ),\n}\n# Exchange-confirmed exceptional closures are separate from the published\n# annual calendar. 2026-07-10: Typhoon Bavi closed both TWSE and TPEx for\n# the entire day. Exchange announcements reported 2026-07-09:\n# https://www.cna.com.tw/news/afe/202607090360.aspx\n_CONFIRMED_AD_HOC_CLOSURES = {\n    date(2026, 7, 10): \'TWSE_TPEX:2026-07-09_typhoon_Bavi:https://www.cna.com.tw/news/afe/202607090360.aspx\',\n}\nEXCHANGE_CLOSURES = frozenset(\n    date.fromisoformat(f\'{year}-{md}\')\n    for year, month_days in _SCHEDULED_CLOSURES.items()\n    for md in month_days\n).union(_CONFIRMED_AD_HOC_CLOSURES)\nTRADING_CALENDAR_SOURCE = \'TWSE:annual_market_holidays_2025_2026\'\n\n\ndef is_scheduled_closure(day: date) -> bool:\n    return day in EXCHANGE_CLOSURES\n\n\n\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef digest(body: bytes) -> str:\n    return hashlib.sha256(body).hexdigest()\n\n\ndef write_atomic(path: Path, data: bytes) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.tmp\')\n    with tmp.open(\'wb\') as f:\n        f.write(data)\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(tmp, path)\n\n\ndef json_bytes(obj: Any) -> bytes:\n    return (json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(\',\', \':\')) + \'\\n\').encode(\'utf-8\')\n\n\ndef number(raw: Any) -> float | None:\n    s = re.sub(r\'<[^>]+>\', \'\', str(raw if raw is not None else \'\')).replace(\',\', \'\').strip()\n    if s in EMPTY_TOKENS:\n        return None\n    try:\n        return float(s)\n    except (ValueError, OverflowError):\n        return None\n\n\ndef shares(raw: Any) -> int | None:\n    n = number(raw)\n    return int(n) if n is not None and n >= 0 and n.is_integer() else None\n\n\ndef company_universe(rows: list[dict], market: str) -> dict[str, dict]:\n    result: dict[str, dict] = {}\n    if not isinstance(rows, list):\n        raise ValueError(\'company master must be JSON/CSV list\')\n    for r in rows:\n        code = str(r.get(\'公司代號\', r.get(\'SecuritiesCompanyCode\', \'\'))).strip()\n        name = str(r.get(\'公司名稱\', r.get(\'CompanyName\', \'\'))).strip()\n        if not re.fullmatch(r\'[1-9][0-9]{3}\', code) or not name or EXCLUDE_NAME.search(name):\n            continue\n        result[code] = {\'symbol\': code, \'market\': market, \'name\': name}\n    if len(result) < 1:\n        raise ValueError(f\'{market} official company master yielded no ordinary-stock candidates\')\n    return result\n\n\ndef csv_company_rows(body: bytes) -> list[dict]:\n    content = None\n    for encoding in (\'utf-8-sig\', \'big5\'):\n        try:\n            content = body.decode(encoding)\n            break\n        except UnicodeDecodeError:\n            pass\n    if content is None:\n        raise ValueError(\'company CSV decode error\')\n    return list(csv.DictReader(io.StringIO(content)))\n\n\ndef _twse_fields_and_rows(payload: dict) -> tuple[list, list]:\n    tables = payload.get(\'tables\') or []\n    for t in tables:\n        f = t.get(\'fields\') or []\n        if \'證券代號\' in f and \'收盤價\' in f and \'成交股數\' in f:\n            return f, t.get(\'data\') or []\n    for idx in (5, 8, 9, 4, 1):\n        fields = payload.get(f\'fields{idx}\') or []\n        if \'證券代號\' in fields and \'收盤價\' in fields and \'成交股數\' in fields:\n            return fields, payload.get(f\'data{idx}\') or []\n    raise ValueError(\'TWSE schema changed: equity table missing\')\n\n\ndef _row(symbol: str, market: str, trading_date: str, fields: dict, observed_at: str, source: str) -> dict | None:\n    close = number(fields.get(\'close\'))\n    vol = shares(fields.get(\'volume\'))\n    if close is None or close <= 0 or vol is None:\n        return None\n    return dict(symbol=symbol, market=market, trade_date=trading_date,\n                open=number(fields.get(\'open\')), high=number(fields.get(\'high\')),\n                low=number(fields.get(\'low\')), close=close, volume=vol,\n                turnover=shares(fields.get(\'turnover\')), source=source, observed_at=observed_at)\n\n\ndef parse_twse(payload: dict, trade_day: date, whitelist: dict[str, dict], observed_at: str) -> list[dict]:\n    fields, data = _twse_fields_and_rows(payload)\n    ix = {str(v).replace(\' \', \'\'): i for i, v in enumerate(fields)}\n    keys = [\'證券代號\', \'成交股數\', \'收盤價\', \'開盤價\', \'最高價\', \'最低價\', \'成交金額\']\n    if not all(k in ix for k in keys):\n        raise ValueError(\'TWSE daily mandatory fields missing\')\n    out = []\n    for item in data:\n        symbol = re.sub(\'<[^>]*>\', \'\', str(item[ix[\'證券代號\']])).strip()\n        if symbol not in whitelist:\n            continue\n        f = {\'open\': item[ix[\'開盤價\']], \'high\': item[ix[\'最高價\']],\n             \'low\': item[ix[\'最低價\']], \'close\': item[ix[\'收盤價\']],\n             \'volume\': item[ix[\'成交股數\']], \'turnover\': item[ix[\'成交金額\']]}\n        r = _row(symbol, \'twse\', trade_day.isoformat(), f, observed_at, \'TWSE:MI_INDEX\')\n        if r:\n            out.append(r)\n    return out\n\n\ndef parse_tpex(payload: dict, trade_day: date, whitelist: dict[str, dict], observed_at: str) -> list[dict]:\n    tables = payload.get(\'tables\') or []\n    if not tables and \'aaData\' in payload:\n        tables = [{\'data\': payload[\'aaData\']}]\n    if not tables:\n        raise ValueError(\'TPEx historical daily data table missing\')\n    out = []\n    for table in tables:\n        fields = [str(f).replace(\' \', \'\') for f in table.get(\'fields\', [])]\n        for values in table.get(\'data\', []):\n            if not isinstance(values, list) or len(values) < 10:\n                continue\n            symbol = re.sub(r\'<[^>]*>\', \'\', str(values[0])).strip()\n            if symbol not in whitelist:\n                continue\n            if all(x in fields for x in (\'收盤\', \'開盤\', \'最高\', \'最低\', \'成交股數\', \'成交金額\')):\n                m = {key: fields.index(key) for key in (\'收盤\', \'開盤\', \'最高\', \'最低\', \'成交股數\', \'成交金額\')}\n                vals = dict(close=values[m[\'收盤\']], open=values[m[\'開盤\']], high=values[m[\'最高\']],\n                            low=values[m[\'最低\']], volume=values[m[\'成交股數\']], turnover=values[m[\'成交金額\']])\n            else:\n                # Legacy official stk_wn1430_result.php table layout.\n                vals = dict(close=values[2], open=values[4], high=values[5],\n                            low=values[6], volume=values[8], turnover=values[9])\n            r = _row(symbol, \'tpex\', trade_day.isoformat(), vals, observed_at, \'TPEx:OTC_DAILY\')\n            if r:\n                out.append(r)\n    return out\n\n\ndef no_trading(payload: dict) -> bool:\n    if not isinstance(payload, dict):\n        return False\n    text = str(payload.get(\'stat\') or payload.get(\'message\') or payload.get(\'msg\') or \'\')\n    return any(word in text for word in (\'沒有符合條件\', \'查無資料\', \'未有資料\', \'休市\', \'很抱歉\', \'No data\', \'NO DATA\'))\n\n\ndef fetch_json(session, url: str, params: dict | None, *, retries: int = 3) -> tuple[Any, str]:\n    for attempt in range(retries):\n        try:\n            r = session.get(url, params=params, timeout=35, headers={\'User-Agent\': \'JanusResearch/0.1 (+personal noncommercial research)\'})\n            if r.status_code in (429, 500, 502, 503, 504):\n                raise RuntimeError(f\'retryable HTTP {r.status_code}\')\n            r.raise_for_status()\n            return r.json(), digest(r.content)\n        except (requests.RequestException, ValueError, RuntimeError):\n            if attempt + 1 == retries:\n                raise\n            time.sleep(min(30, 2 ** attempt * 3))\n    raise AssertionError(\'unreachable\')\n\n\ndef fetch_company_snapshots(session) -> tuple[dict, dict]:\n    l, lhash = fetch_json(session, TWSE_COMPANIES, None)\n    o_source = TPEX_COMPANIES\n    try:\n        o, ohash = fetch_json(session, TPEX_COMPANIES, None)\n        if not isinstance(o, list):\n            raise ValueError(\'TPEx company API not a list\')\n    except (ValueError, requests.RequestException, RuntimeError):\n        r = session.get(TPEX_COMPANIES_CSV, timeout=35)\n        r.raise_for_status()\n        o, ohash = csv_company_rows(r.content), digest(r.content)\n        o_source = TPEX_COMPANIES_CSV\n    return ({\'twse\': company_universe(l, \'twse\'), \'tpex\': company_universe(o, \'tpex\')},\n            {\'twse\': {\'url\': TWSE_COMPANIES, \'sha256\': lhash},\n             \'tpex\': {\'url\': o_source, \'sha256\': ohash}})\n\n\ndef daily_url(market: str, day: date) -> tuple[str, dict]:\n    if market == \'twse\':\n        return TWSE_DAILY, {\'date\': day.strftime(\'%Y%m%d\'), \'type\': \'ALLBUT0999\', \'response\': \'json\'}\n    roc = f\'{day.year - 1911}/{day.month:02d}/{day.day:02d}\'\n    return TPEX_DAILY, {\'l\': \'zh-tw\', \'d\': roc, \'se\': \'EW\'}\n\n\ndef read_chunk(path: Path, universe_hash: str) -> dict:\n    raw = gzip.decompress(path.read_bytes())\n    data = json.loads(raw)\n    if data.get(\'universe_hash\') != universe_hash or data.get(\'schema_version\') != 1:\n        raise ValueError(\'checkpoint universe/schema changed; choose a new revision\')\n    return data\n\n\ndef collect_chunk(session, market: str, dates: list[date], universe: dict, universe_hash: str,\n                  path: Path, *, min_interval: float, request_budget: list[int],\n                  deadline: float | None = None) -> bool:\n    """Resume within a 10-day chunk. Every completed day is saved to Drive.\n\n    Existing full checkpoints from earlier versions remain readable; interrupted\n    checkpoints carry an exact prefix of days and never count as a full chunk.\n    """\n    if path.exists():\n        saved = read_chunk(path, universe_hash)\n        if saved.get(\'market\') != market:\n            raise ValueError(\'checkpoint market changed; choose a new revision\')\n        statuses = saved.get(\'days\')\n        rows = saved.get(\'rows\')\n        if not isinstance(statuses, list) or not isinstance(rows, list):\n            raise ValueError(\'invalid checkpoint content\')\n        expected = [d.isoformat() for d in dates]\n        actual = [d.get(\'day\') if isinstance(d, dict) else None for d in statuses]\n        if actual != expected[:len(actual)]:\n            raise ValueError(\'checkpoint dates/order changed; choose a new revision\')\n        if any(r.get(\'trade_date\') not in set(actual) for r in rows):\n            raise ValueError(\'checkpoint bars do not match completed days\')\n        if len(statuses) == len(dates):\n            return True\n    else:\n        statuses, rows = [], []\n\n    def persist() -> None:\n        content = {\'schema_version\': 1, \'universe_hash\': universe_hash,\n                   \'market\': market, \'days\': statuses, \'rows\': rows}\n        write_atomic(path, gzip.compress(json_bytes(content), mtime=0))\n\n    # Check for budget/time at *each* day, not before an entire 10-day chunk.\n    # Keep a 90s buffer for one request, retries and durable Drive write.\n    for day in dates[len(statuses):]:\n        if is_scheduled_closure(day):\n            statuses.append({\'day\': day.isoformat(), \'status\': \'SCHEDULED_CLOSURE\',\n                             \'calendar_source\': _CONFIRMED_AD_HOC_CLOSURES.get(day, TRADING_CALENDAR_SOURCE)})\n            persist()\n            continue\n        if request_budget[0] <= 0 or (deadline is not None and time.monotonic() >= deadline - 90):\n            return False\n        url, params = daily_url(market, day)\n        payload, source_hash = fetch_json(session, url, params)\n        request_budget[0] -= 1\n        observed = utc_now()\n        if no_trading(payload):\n            statuses.append({\'day\': day.isoformat(), \'status\': \'OFFICIAL_NO_DATA\', \'sha256\': source_hash})\n        else:\n            parsed = parse_twse(payload, day, universe, observed) if market == \'twse\' else parse_tpex(payload, day, universe, observed)\n            if not parsed:\n                raise ValueError(f\'{market} {day} no valid ordinary-stock bars; not treated as holiday\')\n            rows.extend(parsed)\n            statuses.append({\'day\': day.isoformat(), \'status\': \'COLLECTED\',\n                             \'rows\': len(parsed), \'sha256\': source_hash})\n        persist()\n        if min_interval:\n            time.sleep(min_interval)\n    return True\n\ndef month_days(month: str, end: date) -> list[date]:\n    start = date.fromisoformat(month + \'-01\')\n    ny = start.year + (start.month == 12)\n    nm = start.month % 12 + 1\n    stop = min(date(ny, nm, 1), end + timedelta(days=1))\n    arr = []\n    while start < stop:\n        if start.weekday() < 5:\n            arr.append(start)\n        start += timedelta(days=1)\n    return arr\n\n\ndef months_between(start: str, end: str) -> list[str]:\n    s, e = date.fromisoformat(start + \'-01\'), date.fromisoformat(end + \'-01\')\n    if s > e or (e.year - s.year) * 12 + e.month - s.month > 35:\n        raise ValueError(\'month range invalid or exceeds 36 months\')\n    out = []\n    while s <= e:\n        out.append(s.strftime(\'%Y-%m\'))\n        s = date(s.year + (s.month == 12), s.month % 12 + 1, 1)\n    return out\n\n\ndef csv_payload(rows: list[dict], cols=HEADERS) -> bytes:\n    o = io.StringIO(newline=\'\')\n    w = csv.DictWriter(o, fieldnames=list(cols), extrasaction=\'ignore\')\n    w.writeheader()\n    for r in rows:\n        w.writerow(r)\n    return o.getvalue().encode(\'utf-8\')\n\n\ndef finalize_month(root: Path, month: str, revision: str, days: list[date], rosters: dict,\n                   hashes: dict, *, allow_partial: bool = False) -> dict:\n    folder = root / month\n    paths = {m: [folder / \'_checkpoint\' / revision / f\'{m}-{i:03d}.json.gz\'\n                 for i in range(0, len(days), 10)] for m in (\'twse\', \'tpex\')}\n    manifest_path = folder / f\'manifest-{revision}.json\'\n    if manifest_path.exists():\n        existing = json.loads(manifest_path.read_text(\'utf-8\'))\n        for name, sha in existing.get(\'sha256\', {}).items():\n            if not (folder / name).exists() or digest((folder / name).read_bytes()) != sha:\n                raise ValueError(\'published archive corrupted: use new revision, no overwrite\')\n        return {\'status\': \'ALREADY_ARCHIVED\', \'month\': month}\n    # Existence alone is insufficient: interrupted chunks now persist daily.\n    # A month may be sealed only when *all expected days* in both markets exist.\n    incomplete = 0\n    chunks = []\n    for market, parts in paths.items():\n        for index, part in enumerate(parts):\n            expected_days = [d.isoformat() for d in days[index*10:index*10+10]]\n            if not part.exists():\n                incomplete += 1\n                continue\n            chunk = read_chunk(part, hashes[\'universe\'])\n            actual_days = [x.get(\'day\') for x in chunk.get(\'days\', [])]\n            if chunk.get(\'market\') != market or actual_days != expected_days:\n                if actual_days != expected_days[:len(actual_days)] or chunk.get(\'market\') != market:\n                    raise ValueError(\'checkpoint dates/market mismatch\')\n                incomplete += 1\n                continue\n            chunks.append(chunk)\n    if incomplete:\n        return {\'status\': \'PARTIAL\', \'month\': month, \'pending_chunks\': incomplete}\n    rows = [r for ch in chunks for r in ch[\'rows\']]\n    ids = [(r[\'symbol\'], r[\'trade_date\']) for r in rows]\n    if len(ids) != len(set(ids)):\n        raise ValueError(\'duplicate symbol/day across official chunks\')\n    # Whole-company master is latest snapshot, NOT the historical as-of universe.\n    universe_rows = sorted([x for market in (\'twse\', \'tpex\') for x in rosters[market].values()],\n                           key=lambda r: (r[\'market\'], r[\'symbol\']))\n    statuses = [s for ch in chunks for s in ch[\'days\']]\n    rows.sort(key=lambda r: (r[\'symbol\'], r[\'trade_date\']))\n    content = {\n        f\'ohlcv-{revision}.csv.gz\': gzip.compress(csv_payload(rows), mtime=0),\n        f\'universe-{revision}.csv\': csv_payload(universe_rows, (\'symbol\', \'market\', \'name\')),\n    }\n    market_counts = {}\n    for market in (\'twse\', \'tpex\'):\n        actual = len({r[\'symbol\'] for r in rows if r[\'market\'] == market})\n        market_counts[market] = {\'snapshot_symbols\': len(rosters[market]), \'symbols_with_bars\': actual,\n                                 \'observed_ratio\': actual / max(1, len(rosters[market]))}\n    quality = {\'status\': \'PROBE\', \'market_counts\': market_counts,\n               \'historical_pit\': \'NOT_VERIFIED\', \'historical_survivorship\': \'CURRENT_MASTER_ONLY\',\n               \'research_70pct_certified\': False, \'missing_financials\': \'UNAVAILABLE\',\n               \'corporate_actions\': \'UNADJUSTED\', \'market_calendar\': TRADING_CALENDAR_SOURCE + \'_AND_EXPLICIT_OFFICIAL_NO_DATA\',\n               \'scheduled_closure_days\': sum(s[\'status\'] == \'SCHEDULED_CLOSURE\' for s in statuses),\n               \'collected_days\': sum(s[\'status\'] == \'COLLECTED\' for s in statuses),\n               \'no_data_days\': sum(s[\'status\'] == \'OFFICIAL_NO_DATA\' for s in statuses)}\n    content[f\'quality-{revision}.json\'] = json_bytes(quality)\n    manifest = {\'schema_version\': 1, \'month\': month, \'revision\': revision,\n                \'kind\': \'COLAB_RESEARCH_PROBE\', \'publication_allowed\': False,\n                \'source_urls\': [TWSE_DAILY, TPEX_DAILY, TWSE_COMPANIES, TPEX_COMPANIES],\n                \'source_master_hash\': hashes[\'universe\'], \'captured_master_at\': hashes[\'captured_at\'],\n                \'source_daily_sha256\': statuses,\n                \'sha256\': {name: digest(data) for name, data in content.items()},\n                \'created_at\': utc_now()}\n    for name, body in content.items():\n        if (folder / name).exists():\n            raise ValueError(\'orphan published member; manual inspection before new revision\')\n        write_atomic(folder / name, body)\n    write_atomic(manifest_path, json_bytes(manifest))\n    for name, sha in manifest[\'sha256\'].items():\n        if digest((folder / name).read_bytes()) != sha:\n            raise ValueError(\'published monthly hash verification failed; checkpoints retained\')\n    # Reduce Drive small objects only after full month and hash verification.\n    for parts in paths.values():\n        for chunk in parts:\n            chunk.unlink()\n    (folder / \'_checkpoint\' / revision).rmdir()\n    (folder / \'_checkpoint\').rmdir()\n    return {\'status\': \'PROBE_ARCHIVED\', \'month\': month, \'rows\': len(rows),\n            \'market_counts\': market_counts, \'sha256\': digest(json_bytes(manifest))}\n\n\ndef execute(root: Path, start_month: str, end_month: str, revision: str,\n            *, max_requests: int = 120, min_interval: float = 2.0, session=None,\n            today: date | None = None, max_minutes: float | None = None) -> list[dict]:\n    if not re.fullmatch(r\'r[1-9][0-9]{0,2}\', revision):\n        raise ValueError(\'revision must be r1..r999\')\n    today = today or datetime.now(ZoneInfo("Asia/Taipei")).date()\n    allmonths = months_between(start_month, end_month)\n    if date.fromisoformat(end_month + \'-01\') >= today.replace(day=1):\n        raise ValueError(\'only fully closed historical months are supported; wait until next month\')\n    if max_requests < 0 or max_requests > 500:\n        raise ValueError(\'max_requests must be bounded between 0 and 500\')\n    if max_minutes is not None and not (2 <= max_minutes <= 60):\n        raise ValueError(\'max_minutes must be between 2 and 60\')\n    deadline = None if max_minutes is None else time.monotonic() + max_minutes * 60\n    root.mkdir(parents=True, exist_ok=True)\n    session = session or requests.Session()\n    ctl = root / \'_control\'\n    # Snapshot once per research revision: retries never silently change universe.\n    snap = ctl / f\'universe-{revision}.json\'\n    if snap.exists():\n        cached = json.loads(snap.read_text(\'utf-8\'))\n        universe = cached[\'data\']\n        hashes = {\'universe\': digest(json_bytes(cached)), \'captured_at\': cached[\'captured_at\']}\n    else:\n        rosters, provenance = fetch_company_snapshots(session)\n        cached = {\'captured_at\': utc_now(), \'data\': rosters, \'provenance\': provenance,\n                  \'historical_survivorship\': \'CURRENT_MASTER_ONLY\'}\n        write_atomic(snap, json_bytes(cached))\n        universe = rosters\n        hashes = {\'universe\': digest(json_bytes(cached)), \'captured_at\': cached[\'captured_at\']}\n    budget = [max_requests]\n    outputs = []\n    for month in allmonths:\n        existing_manifests=list((root/month).glob(\'manifest-r[0-9]*.json\'))\n        if existing_manifests:\n            # A finalized earlier revision is authoritative; never re-collect an\n            # archived month just because the annual roster revision rotated.\n            published=max(existing_manifests,key=lambda p:int(re.fullmatch(r\'manifest-r(\\d+)\\.json\',p.name).group(1)))\n            existing = json.loads(published.read_text(\'utf-8\'))\n            if all((root / month / name).exists() and digest((root / month / name).read_bytes()) == sha\n                   for name, sha in existing[\'sha256\'].items()):\n                outputs.append({\'status\': \'ALREADY_ARCHIVED\', \'month\': month})\n                continue\n            raise ValueError(\'finalized monthly file failed checksum; no silent overwrite\')\n        days = month_days(month, today - timedelta(days=1))\n        if not days:\n            outputs.append({\'status\': \'NOT_MATURED\', \'month\': month})\n            continue\n        for market in (\'twse\', \'tpex\'):\n            for i in range(0, len(days), 10):\n                path = root / month / \'_checkpoint\' / revision / f\'{market}-{i:03d}.json.gz\'\n                if not collect_chunk(session, market, days[i:i+10], universe[market], hashes[\'universe\'],\n                                     path, min_interval=min_interval, request_budget=budget,\n                                     deadline=deadline):\n                    break\n        outputs.append(finalize_month(root, month, revision, days, universe, hashes))\n        if budget[0] == 0 or (deadline is not None and time.monotonic() >= deadline - 90):\n            break\n    return outputs\n\n\ndef main() -> None:\n    p = argparse.ArgumentParser(description=\'Official TWSE/TPEx Colab historical common-stock pilot; no model publish\')\n    p.add_argument(\'--drive-root\', required=True, type=Path, help=\'mounted private janusChatGPT/資料源\')\n    p.add_argument(\'--start\', required=True, help=\'YYYY-MM\')\n    p.add_argument(\'--end\', required=True, help=\'YYYY-MM\')\n    p.add_argument(\'--revision\', default=\'r1\')\n    p.add_argument(\'--max-requests\', type=int, default=120)\n    p.add_argument(\'--interval\', type=float, default=2.0)\n    p.add_argument(\'--max-minutes\', type=float, default=10.0,\n                   help=\'timebox each manual Colab session, 2-60 minutes (default 10)\')\n    a = p.parse_args()\n    if a.interval < 1.0:\n        p.error(\'interval must be >= 1 second\')\n    result = execute(a.drive_root, a.start, a.end, a.revision,\n                     max_requests=a.max_requests, min_interval=a.interval,\n                     max_minutes=a.max_minutes)\n    for item in result:\n        print(json.dumps(item, ensure_ascii=False))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'official_batch.py': '"""Official batch-first research collection. No network activity at import."""\nfrom __future__ import annotations\nimport gzip\nimport html\nimport io\nimport json\nimport re\nimport time\nfrom datetime import date, timedelta\nfrom pathlib import Path\nimport pandas as pd\nimport requests\nfrom research_collection import atomic, encode, sha, now, clean_number, Collector, build_tasks, normalise\n\nVERSION = \'3.0.0\'\nMARKETS = {\'twse\': \'sii\', \'tpex\': \'otc\'}\nFINANCIAL = {\'income\': \'t163sb04\', \'balance\': \'t163sb05\', \'cashflow\': \'t163sb20\'}\n\ndef label(value):\n    if isinstance(value, tuple):\n        value = next((str(x) for x in reversed(value) if not str(x).startswith(\'Unnamed\')), \'\')\n    return re.sub(r\'\\s+\', \'\', str(value)).replace(\'\\ufeff\', \'\')\n\ndef decoded(body):\n    for encoding in (\'utf-8-sig\', \'cp950\'):\n        try:\n            return body.decode(encoding)\n        except UnicodeDecodeError:\n            pass\n    raise ValueError(\'UNKNOWN_ENCODING\')\n\ndef csv_table(body):\n    text = decoded(body)\n    lines = text.splitlines()\n    start = next((i for i, s in enumerate(lines) if \'公司代號\' in s and \'當月營收\' in s), None)\n    if start is not None:\n        return pd.read_csv(io.StringIO(\'\\n\'.join(lines[start:])), dtype=str, keep_default_na=False)\n    # Backup official archive presentation for months whose CSV output is malformed.\n    if \'<table\' in text.lower():\n        for table in pd.read_html(io.StringIO(text), keep_default_na=False):\n            labels = [label(c) for c in table.columns]\n            if \'公司代號\' in labels and any(\'當月營收\' in x for x in labels):\n                table.columns = labels\n                return table.astype(str)\n    raise ValueError(\'REVENUE_HEADER_NOT_FOUND\')\n\ndef base_row(task, symbol):\n    return dict(symbol=symbol, market=task[\'market\'], dataset=task[\'kind\'],\n                partition_month=task[\'period\'][:7], source_id=task[\'source\'],\n                observed_at=now(), available_at=None, published_at=None,\n                historical_pit=\'NOT_VERIFIED\', publication_allowed=False)\n\ndef revenue_rows(body, task, roster):\n    frame = csv_table(body)\n    frame.columns = [label(x) for x in frame.columns]\n    def column(candidates):\n        return next((c for c in frame.columns if c in candidates), None)\n    sid = column([\'公司代號\'])\n    amount = column([\'營業收入-當月營收\', \'當月營收\'])\n    period = column([\'資料年月\'])\n    if not all((sid, amount, period)):\n        raise ValueError(\'REVENUE_SCHEMA_OR_PERIOD_MISSING\')\n    result, matched_period = [], 0\n    year, month = map(int, task[\'period\'].split(\'-\'))\n    # Historical MOPS archives sometimes include aggregate/footer rows or mixed periods.\n    # Reject mismatched rows, NOT the entire official month batch.\n    acceptable = {f\'{year}{month:02d}\', f\'{year-1911}{month:02d}\',\n                  f\'{year-1911}{month}\', f\'{year}{month}\'}\n    for _, r in frame.iterrows():\n        symbol = str(r[sid]).strip()\n        if not re.fullmatch(r\'\\d{4}\', symbol):\n            continue\n        digits = re.sub(r\'\\D\', \'\', str(r[period]))\n        if digits not in acceptable:\n            continue\n        matched_period += 1\n        if roster.get(symbol, {}).get(\'market\') != task[\'market\']:\n            continue\n        value = clean_number(r[amount])\n        if value is None or value < 0:\n            continue\n        row = base_row(task, symbol)\n        row.update(fiscal_month=task[\'period\'], revenue_twd=value*1000,\n                   raw_revenue_thousand_twd=value, unit_basis=\'TWD_THOUSAND\')\n        result.append(row)\n    if not matched_period:\n        raise ValueError(\'REVENUE_NO_MATCHING_PERIOD\')\n    return result\n\ndef normal_date(value):\n    parts = re.findall(r\'\\d+\', str(value))\n    if len(parts) == 1 and len(parts[0]) == 8:\n        return date.fromisoformat(f\'{parts[0][:4]}-{parts[0][4:6]}-{parts[0][6:]}\').isoformat()\n    if len(parts) == 3:\n        y, m, d = map(int, parts)\n        return date(y+1911 if y < 1911 else y, m, d).isoformat()\n    raise ValueError(\'RESPONSE_DATE_MISSING\')\n\ndef _valuation_entry(task, roster, day, symbol, pe_value, pb_value, yield_value):\n    symbol = str(symbol).strip()\n    if not re.fullmatch(r\'\\d{4}\', symbol) or roster.get(symbol, {}).get(\'market\') != task[\'market\']:\n        return None\n    row = base_row(task, symbol)\n    for key, raw in [(\'pe_ratio\', pe_value), (\'pb_ratio\', pb_value),\n                     (\'dividend_yield_percent\', yield_value)]:\n        value = clean_number(raw)\n        row[key] = value if value is not None and value > 0 else (0.0 if key == \'dividend_yield_percent\' and value == 0 else None)\n    row.update(trade_date=day, sampling=\'MONTH_END_SNAPSHOT\')\n    return row\n\n\ndef tpex_html_valuation_rows(body, task, roster, requested_date):\n    """TPEx\'s historical peratio_analysis HTML export, one daily all-stock table.\n\n    A provider\'s current/default HTML response must never be passed off as a\n    historical response. Require a matching DATE label before reading any row.\n    """\n    if task[\'market\'] != \'tpex\':\n        raise ValueError(\'TPEx_HTML_WRONG_MARKET\')\n    raw_text = decoded(body)\n    flat = html.unescape(re.sub(r\'<[^>]+>\', \' \', raw_text))\n    if re.search(r\'Access Denied|Forbidden|Too Many Requests|CAPTCHA\', flat, re.IGNORECASE):\n        raise RuntimeError(\'SOURCE_BLOCKED_CONTENT\')\n    match = re.search(r\'(?:資料日期|報表日期)\\s*[：:]?\\s*(\\d{2,4}[/-]\\d{1,2}[/-]\\d{1,2})\', flat)\n    if not match:\n        # A holiday/no-quote HTML response may contain no date stamp at all.\n        # It is safe to *skip* it, never to ingest it as historical data.\n        if re.search(r\'查無|沒有資料|無符合|無資料\', flat):\n            return []\n        raise ValueError(\'TPEx_HTML_DATE_NOT_CONFIRMED\')\n    try:\n        observed = normal_date(match.group(1))\n    except ValueError as exc:\n        raise ValueError(\'TPEx_HTML_DATE_NOT_CONFIRMED\') from exc\n    if observed != requested_date:\n        raise ValueError(\'RESPONSE_DATE_MISMATCH\')\n    # Actual historical provider HTML sometimes returns a valid dated table\n    # whose tbody is empty and tfoot explicitly says 共0筆. Treat only this\n    # strict combination as a no-quote day, then let collect_one step back.\n    # Never treat a date mismatch or unknown HTML schema as no-data.\n    has_empty_tbody = bool(re.search(r\'<tbody\\b[^>]*>\\s*</tbody\\s*>\', raw_text, re.IGNORECASE))\n    count_zero = bool(re.search(r\'共\\s*0\\s*筆\', flat))\n    if has_empty_tbody and count_zero:\n        return []\n    if has_empty_tbody != count_zero:\n        raise ValueError(\'TPEx_HTML_EMPTY_STATUS_INCONSISTENT\')\n    try:\n        tables = pd.read_html(io.StringIO(raw_text), keep_default_na=False)\n    except (ValueError, ImportError) as exc:\n        raise ValueError(\'TPEx_HTML_TABLE_NOT_FOUND\') from exc\n    results = []\n    for frame in tables:\n        cols = [label(c) for c in frame.columns]\n        # Some official exports present the column heading in the first row.\n        def positions(names):\n            code = next((i for i, x in enumerate(names) if x in (\'股票代號\',\'證券代號\',\'股票代碼\',\'代號\',\'代碼\')), None)\n            pe = next((i for i, x in enumerate(names) if \'本益比\' in x), None)\n            pb = next((i for i, x in enumerate(names) if \'股價淨值比\' in x), None)\n            dy = next((i for i, x in enumerate(names) if \'殖利率\' in x), None)\n            return (code, pe, pb, dy)\n        indices = positions(cols)\n        if None in indices:\n            for n, values in enumerate(frame.head(6).itertuples(index=False, name=None)):\n                candidate = [label(c) for c in values]\n                indices = positions(candidate)\n                if None not in indices:\n                    frame = frame.iloc[n+1:]\n                    break\n        if None in indices:\n            continue\n        code, pe, pb, dy = indices\n        for values in frame.itertuples(index=False, name=None):\n            try:\n                row = _valuation_entry(task, roster, requested_date, values[code],\n                                       values[pe], values[pb], values[dy])\n            except (IndexError, TypeError, ValueError):\n                continue\n            if row is not None:\n                results.append(row)\n    if results:\n        unique = {r[\'symbol\']: r for r in results}\n        return list(unique.values())\n    # Distinguish a historical day with no quotes from a changed response format.\n    if re.search(r\'查無|沒有資料|無符合|無資料\', flat):\n        return []\n    raise ValueError(\'TPEx_HTML_SCHEMA_UNRECOGNISED\')\n\n\ndef valuation_rows(body, task, roster, requested_date):\n    if decoded(body).lstrip().startswith(\'<\'):\n        return tpex_html_valuation_rows(body, task, roster, requested_date)\n    obj = json.loads(decoded(body))\n    returned = obj.get(\'date\') or obj.get(\'reportDate\')\n    if returned is None:\n        raise ValueError(\'RESPONSE_DATE_MISSING\')\n    if normal_date(returned) != requested_date:\n        raise ValueError(\'RESPONSE_DATE_MISMATCH\')\n    tables = obj.get(\'tables\') or [obj]\n    result = []\n    for table in tables:\n        fields = [label(x) for x in table.get(\'fields\', [])]\n        rows = table.get(\'data\', table.get(\'aaData\', []))\n        if not fields and rows:\n            if task[\'market\'] == \'tpex\' and len(rows[0]) >= 7:\n                fields = [\'代號\',\'名稱\',\'本益比\',\'每股股利\',\'股利年度\',\'殖利率(%)\',\'股價淨值比\']\n        code = next((i for i,c in enumerate(fields) if c in (\'證券代號\',\'代號\',\'股票代號\')), None)\n        pe = next((i for i,c in enumerate(fields) if c == \'本益比\'), None)\n        pb = next((i for i,c in enumerate(fields) if c == \'股價淨值比\'), None)\n        dy = next((i for i,c in enumerate(fields) if c.startswith(\'殖利率\')), None)\n        if None in (code, pe, pb, dy):\n            continue\n        for values in rows:\n            row = _valuation_entry(task, roster, requested_date, values[code],\n                                   values[pe], values[pb], values[dy])\n            if row is not None:\n                result.append(row)\n    return result\n\ndef financial_rows(body, task, roster):\n    text = decoded(body)\n    year, quarter = int(task[\'period\'][:4]), int(task[\'period\'][-1])\n    # Never label a latest/default response as the requested historical quarter.\n    flat = html.unescape(re.sub(r\'<[^>]+>\', \' \', text))\n    confirmed_year = bool(re.search(rf\'(?<!\\d)(?:{year-1911}|{year})(?:\\s*年|\\s*年度|\\s*[/-])\', flat))\n    confirmed_quarter = bool(re.search(rf\'(?:第\\s*)?0?{quarter}\\s*季|第\\s*0?{quarter}\\s*季\', flat))\n    period_confirmed = confirmed_year and confirmed_quarter\n    # A period supplied in the request is usable for exploratory research even\n    # if the MOPS HTML omits it. It is NOT PIT certified or publishable.\n    result = []\n    for table_no, frame in enumerate(pd.read_html(io.StringIO(text), keep_default_na=False)):\n        fields = [label(x) for x in frame.columns]\n        sid = next((i for i,c in enumerate(fields) if \'公司代號\' in c or \'公司代碼\' in c), None)\n        if sid is None:\n            for index, values in enumerate(frame.head(10).itertuples(index=False,name=None)):\n                candidate=[label(v) for v in values]\n                sid=next((i for i,c in enumerate(candidate) if c in (\'公司代號\',\'公司代碼\')),None)\n                if sid is not None:\n                    fields=candidate\n                    frame=frame.iloc[index+1:]\n                    break\n        if sid is None:\n            continue\n        for values in frame.itertuples(index=False, name=None):\n            symbol = str(values[sid]).strip()\n            if roster.get(symbol, {}).get(\'market\') != task[\'market\']:\n                continue\n            for i, item in enumerate(fields):\n                if i == sid or item in (\'公司名稱\',\'公司簡稱\'):\n                    continue\n                try:\n                    value = clean_number(values[i])\n                except (ValueError, TypeError):\n                    continue\n                if value is None:\n                    continue\n                row = base_row(task, symbol)\n                row.update(partition_month=f\'{year}-{quarter*3:02d}\', fiscal_quarter=task[\'period\'],\n                           item=item, value=value, table_no=table_no,\n                           period_basis=(\'OFFICIAL_REPORTED\' if period_confirmed else \'QUERY_PERIOD_UNVERIFIED\'),\n                           unit_basis=\'RETAIN_SOURCE_HEADER_VERIFY_BEFORE_TRAINING\',\n                           period_confirmed=period_confirmed)\n                result.append(row)\n    return result\n\ndef make_tasks(end_month, revenue_months=48, valuation_months=36, financial_quarters=16,\n               revenue_first=\'2022-10\', valuation_first=\'2023-10\', financial_first=\'2022Q4\'):\n    # Anchored earliest history: a future run MUST NOT drop old official input months\n    # from the model-facing Parquet merely because a 36/48-month window advanced.\n    end = pd.Period(end_month, \'M\')\n    revenue_months = max(revenue_months, int(end.ordinal - pd.Period(revenue_first, \'M\').ordinal + 1))\n    valuation_months = max(valuation_months, int(end.ordinal - pd.Period(valuation_first, \'M\').ordinal + 1))\n    tasks = []\n    for kind, count in [(\'revenue\',revenue_months),(\'valuation\',valuation_months)]:\n        for p in pd.period_range(end=end, periods=count, freq=\'M\'):\n            for market in MARKETS:\n                tasks.append(dict(kind=kind,period=str(p),market=market,\n                                  source=\'mops\' if kind==\'revenue\' else market))\n    # Include only completed quarters; announcements can still be pending.\n    qend = end.asfreq(\'Q\')\n    if qend.end_time.date() > end.end_time.date():\n        qend -= 1\n    financial_quarters = max(financial_quarters, int(qend.ordinal - pd.Period(financial_first, \'Q\').ordinal + 1))\n    for p in pd.period_range(end=qend, periods=financial_quarters, freq=\'Q\'):\n        for kind in FINANCIAL:\n            for market in MARKETS:\n                tasks.append(dict(kind=kind,period=str(p),market=market,source=\'mops\'))\n    return tasks\n\nclass BatchCollector:\n    def __init__(self, root, roster, *, max_requests=160, max_minutes=45, interval=3,\n                 session=None, clock=time.monotonic, sleep=time.sleep):\n        self.root, self.roster = Path(root), roster\n        self.base = self.root/\'_official_batch\'/\'v2\'\n        self.session = session or requests.Session()\n        self.clock, self.sleep = clock, sleep\n        self.deadline = clock()+max_minutes*60\n        self.remaining, self.interval, self.last = max_requests, interval, None\n        self.errors = []\n        self.budget_stop = False\n        self.resume_task = None\n\n    def path(self, task):\n        # Single dataset package: both original OHLCV and new official batches\n        # are co-located in data/YYYY-MM (quarter data in data/YYYYQn).\n        partition=task[\'period\']\n        if \'Q\' in partition:\n            partition=f"{partition[:4]}-{int(partition[-1])*3:02d}"\n        return self.root/\'data\'/partition/f"official-{task[\'kind\']}-{task[\'market\']}.json.gz"\n\n    def load(self, task):\n        p = self.path(task)\n        if not p.exists():\n            # Initial v3 migration moved historical v2 months intact.\n            p = self.root/\'data\'/task[\'period\']/f"{task[\'kind\']}-{task[\'market\']}.json.gz"\n        if not p.exists():\n            p = self.base/task[\'period\']/f"{task[\'kind\']}-{task[\'market\']}.json.gz"\n        if not p.exists():\n            return None\n        record = json.loads(gzip.decompress(p.read_bytes()))\n        if record[\'task\'] != task or sha(encode(record[\'rows\'])) != record[\'rows_sha256\']:\n            raise ValueError(\'BATCH_CHECKSUM_OR_TASK_MISMATCH\')\n        raw = self.base/record[\'raw_path\']\n        if sha(raw.read_bytes()) != record[\'response_sha256\']:\n            raise ValueError(\'RAW_CHECKSUM_MISMATCH\')\n        return record\n\n    def fetch(self, url, *, params=None, data=None):\n        if self.remaining <= 0 or self.clock()+50 >= self.deadline:\n            raise RuntimeError(\'BUDGET_STOP\')\n        if self.last is not None:\n            self.sleep(max(0, self.interval-(self.clock()-self.last)))\n        self.last = self.clock()\n        self.remaining -= 1\n        kwargs = dict(timeout=40, headers={\'User-Agent\':\'JanusResearch/2.0\'}, params=params)\n        response = (self.session.post(url, data=data, **kwargs) if data is not None\n                    else self.session.get(url, **kwargs))\n        if response.status_code in (401,403,429):\n            raise RuntimeError(\'SOURCE_BLOCKED_\'+str(response.status_code))\n        response.raise_for_status()\n        # Persist every raw response, including schema errors, by content hash.\n        raw_path = \'raw/\'+sha(response.content)+\'.bin\'\n        if not (self.base/raw_path).exists():\n            atomic(self.base/raw_path, response.content)\n        return response.content, raw_path\n\n    def collect_one(self, task):\n        kind, market, period = task[\'kind\'], task[\'market\'], task[\'period\']\n        params, data = None, None\n        if kind == \'revenue\':\n            y,m = map(int,period.split(\'-\'))\n            url = f\'https://mopsov.twse.com.tw/nas/t21/{MARKETS[market]}/t21sc03_{y-1911}_{m}.csv\'\n        elif kind in FINANCIAL:\n            url = \'https://mopsov.twse.com.tw/mops/web/ajax_\'+FINANCIAL[kind]\n            data = dict(encodeURIComponent=1, step=1, firstin=1, off=1, isQuery=\'Y\',\n                        TYPEK=MARKETS[market], year=int(period[:4])-1911, season=\'0\'+period[-1])\n        else:\n            # Try up to ten calendar days backwards. Cache progress between runs.\n            end = pd.Period(period,\'M\').end_time.date()\n            progress_path = self.path(task).with_suffix(\'.attempts.json\')\n            tried = json.loads(progress_path.read_text(\'utf-8\')) if progress_path.exists() else []\n            for offset in range(10):\n                day = end-timedelta(days=offset)\n                if day.weekday() >= 5 or day.isoformat() in tried:\n                    continue\n                if market == \'twse\':\n                    url = \'https://www.twse.com.tw/rwd/zh/afterTrading/BWIBBU_d\'\n                    params = dict(date=day.strftime(\'%Y%m%d\'),selectType=\'ALL\',response=\'json\')\n                else:\n                    # TPEx publishes a dated HTML all-stock export. The old\n                    # o=json response can be an unrecognised empty envelope.\n                    # Prefer the documented HTML format rather than performing\n                    # thousands of per-stock FinMind requests unnecessarily.\n                    url = \'https://www.tpex.org.tw/web/stock/aftertrading/peratio_analysis/pera_result.php\'\n                    params = dict(l=\'zh-tw\',d=f\'{day.year-1911}/{day:%m/%d}\',o=\'htm\',c=\'\',s=\'0,asc\')\n                body, raw_path = self.fetch(url,params=params)\n                if market == \'tpex\':\n                    rows = tpex_html_valuation_rows(body,task,self.roster,day.isoformat())\n                    if rows:\n                        return rows, body, raw_path, url, params\n                    # Only a dated explicit no-data response allows previous day.\n                    tried.append(day.isoformat())\n                    atomic(progress_path,encode(tried))\n                    continue\n                obj = json.loads(decoded(body))\n                has_data = any(t.get(\'data\') or t.get(\'aaData\') for t in (obj.get(\'tables\') or [obj]))\n                if not has_data:\n                    status = str(obj.get(\'stat\',\'\'))\n                    if \'沒有\' not in status and \'查無\' not in status and not (\'aaData\' in obj and obj[\'aaData\']==[]):\n                        raise ValueError(\'VALUATION_UNRECOGNISED_EMPTY\')\n                    tried.append(day.isoformat())\n                    atomic(progress_path,encode(tried))\n                    continue\n                rows = valuation_rows(body,task,self.roster,day.isoformat())\n                return rows, body, raw_path, url, params\n            raise ValueError(\'MONTH_END_NO_VALID_DATA\')\n        body, raw_path = self.fetch(url,params=params,data=data)\n        if kind == \'revenue\':\n            try:\n                rows = revenue_rows(body,task,self.roster)\n            except ValueError as e:\n                if str(e) not in (\'REVENUE_HEADER_NOT_FOUND\',\'REVENUE_NO_MATCHING_PERIOD\'):\n                    raise\n                # Historical official HTML variant (not a third-party scrape).\n                alt = url[:-4] + \'_0.html\'\n                body, raw_path = self.fetch(alt)\n                rows = revenue_rows(body,task,self.roster)\n                url, data = alt, None\n        else:\n            rows = financial_rows(body,task,self.roster)\n        return rows, body, raw_path, url, data\n\n    def run(self, tasks):\n        failures = {}\n        blocked_sources=set()\n        for task in tasks:\n            if self.load(task):\n                continue\n            if self.remaining <= 0 or self.clock()+50 >= self.deadline:\n                self.budget_stop = True\n                self.resume_task = task\n                break\n            # A broken adapter does not trigger hundreds of identical failed requests.\n            key = (task[\'kind\'],task[\'market\'])\n            if failures.get(key,0) >= 2 or task[\'source\'] in blocked_sources:\n                continue\n            try:\n                rows,body,raw_path,url,parameters = self.collect_one(task)\n                if not rows:\n                    raise ValueError(\'NO_ELIGIBLE_ROWS\')\n                keys = [(r[\'symbol\'], r.get(\'item\',\'\'),r.get(\'table_no\',0)) for r in rows]\n                if len(keys) != len(set(keys)):\n                    raise ValueError(\'DUPLICATE_KEYS\')\n                record = dict(task=task, rows=rows, rows_sha256=sha(encode(rows)), raw_path=raw_path,\n                              response_sha256=sha(body),source_url=url,request_parameters=parameters,\n                              collected_at=now(),status=\'COLLECTED\',version=VERSION)\n                atomic(self.path(task),gzip.compress(encode(record),mtime=0))\n                print(task[\'period\'],task[\'kind\'],task[\'market\'],len(rows))\n            except (ValueError, requests.RequestException, RuntimeError, KeyError) as exc:\n                reason = str(exc)[:160] if not isinstance(exc,requests.RequestException) else type(exc).__name__\n                if reason == \'BUDGET_STOP\':\n                    self.budget_stop = True\n                    self.resume_task = task\n                    break\n                self.errors.append(dict(task=task,error=reason,at=now()))\n                failures[key] = failures.get(key,0)+1\n                if reason.startswith(\'SOURCE_BLOCKED_\'):\n                    blocked_sources.add(task[\'source\'])\n                elif isinstance(exc, RuntimeError):\n                    break\n        atomic(self.base/\'last_errors.json\',encode(self.errors))\n        pending=sum(self.load(t) is None for t in tasks)\n        self.run_state=dict(status=\'BUDGET_PAUSED\' if self.budget_stop else (\'PARTIAL\' if pending else \'ALL_SAVED\'),\n                            requested_batches=len(tasks),pending_batches=pending,requests_remaining=self.remaining,\n                            resume_task=self.resume_task,run_at=now(),errors=len(self.errors))\n        atomic(self.base/\'last_run_state.json\',encode(self.run_state))\n        return self.errors\n\ndef legacy_records(root):\n    """Read all old plan batches once; fail loudly on corrupt checkpoints."""\n    records, seen = [],set()\n    for path in sorted((Path(root)/\'_collection\').glob(\'*/batch-*.json.gz\')):\n        block = json.loads(gzip.decompress(path.read_bytes()))\n        for rec in block[\'records\'].values():\n            if sha(encode(rec[\'raw_rows\'])) != rec[\'raw_rows_sha256\']:\n                raise ValueError(\'LEGACY_CHECKPOINT_HASH_MISMATCH: \'+str(path))\n            if rec[\'status\'] == \'COLLECTED\':\n                key=sha(encode(dict(task=rec[\'task\'],raw=rec[\'raw_rows_sha256\'],at=rec[\'observed_at\'])))\n                if key not in seen:\n                    records.append(rec)\n                    seen.add(key)\n    return records\n\ndef all_rows(batch, tasks, old):\n    rows = []\n    for task in tasks:\n        rec = batch.load(task)\n        if rec:\n            rows.extend(rec[\'rows\'])\n    for rec in old:\n        rows.extend(dict(r,dataset=rec[\'task\'][\'kind\']) for r in rec[\'rows\'])\n    return rows\n\ndef period_of(row, kind):\n    if kind == \'revenue\':\n        return row.get(\'fiscal_month\')\n    if kind in FINANCIAL:\n        if row.get(\'fiscal_quarter\'):\n            return row[\'fiscal_quarter\']\n        # Provider finance dates/period basis remain unverified; never use them to close official gaps.\n        return None\n    return row.get(\'partition_month\')\n\n\n# 2023-10 TWSE valuation: audited omissions from the *current-roster* denominator.\n# This is a bounded research-cohort correction, NOT a certified historical PIT universe.\n_HISTORICAL_ELIGIBILITY_AUDIT = {\n    (\'valuation\', \'twse\', \'2023-10\'): {\n        \'file\': \'2023-10_TWSE_112檔歷史資格核查.csv\',\n        \'sha256\': \'ec4f74fbafeab49cebe95dc6e0b5f3bd38d66d339027f9b71989b2f93290b9a9\',\n        \'period_end\': \'2023-10-31\',\n        \'rows\': 112,\n    }\n}\n\n\ndef audited_eligible_roster(root, kind, market, period, expected):\n    """Fail closed if the verified, narrowly scoped audit cannot be reproduced."""\n    expected = set(expected)\n    key = (kind, market, period)\n    spec = _HISTORICAL_ELIGIBILITY_AUDIT.get(key)\n    if spec is None:\n        return expected, None\n    from pathlib import Path\n    import hashlib\n    import pandas as pd\n    path = Path(root) / \'reports\' / spec[\'file\']\n    if not path.is_file():\n        raise RuntimeError(\'HISTORICAL_ROSTER_AUDIT_MISSING: \'+str(path))\n    if hashlib.sha256(path.read_bytes()).hexdigest() != spec[\'sha256\']:\n        raise RuntimeError(\'HISTORICAL_ROSTER_AUDIT_SHA256_MISMATCH: \'+str(path))\n    audit = pd.read_csv(path, dtype=str, encoding=\'utf-8-sig\').fillna(\'\')\n    required = {\'股票代碼\',\'評估期末\',\'官方上市日期\',\'規範化原因\',\'證據網址\'}\n    if not required.issubset(audit.columns) or len(audit) != spec[\'rows\'] or audit[\'股票代碼\'].duplicated().any():\n        raise RuntimeError(\'HISTORICAL_ROSTER_AUDIT_INVALID_SHAPE\')\n    reasons = {\'FIRST_TWSE_LISTING_AFTER_PERIOD\',\'SECURITY_TYPE_TDR_NOT_ORDINARY_SHARE\'}\n    if set(audit[\'規範化原因\']) != reasons or not (audit[\'評估期末\']==spec[\'period_end\']).all():\n        raise RuntimeError(\'HISTORICAL_ROSTER_AUDIT_INVALID_BASIS\')\n    for record in audit.to_dict(\'records\'):\n        symbol = record[\'股票代碼\']\n        date = record[\'官方上市日期\']\n        reason = record[\'規範化原因\']\n        url = record[\'證據網址\']\n        if not symbol or not date or not url.startswith(\'https://\'):\n            raise RuntimeError(\'HISTORICAL_ROSTER_AUDIT_INCOMPLETE: \'+symbol)\n        if reason == \'FIRST_TWSE_LISTING_AFTER_PERIOD\' and not date > spec[\'period_end\']:\n            raise RuntimeError(\'HISTORICAL_ROSTER_AUDIT_BAD_LISTING_DATE: \'+symbol)\n        if reason == \'SECURITY_TYPE_TDR_NOT_ORDINARY_SHARE\' and not date <= spec[\'period_end\']:\n            raise RuntimeError(\'HISTORICAL_ROSTER_AUDIT_BAD_TDR_DATE: \'+symbol)\n    excluded = set(audit[\'股票代碼\'])\n    if not excluded.issubset(expected):\n        raise RuntimeError(\'HISTORICAL_ROSTER_AUDIT_NOT_SUBSET_OF_COHORT\')\n    # If a subsequent archive changes the initial company snapshot, recompute the\n    # full historical cohort instead of reusing this 2026 snapshot proof blindly.\n    if len(expected) != 1089:\n        raise RuntimeError(\'HISTORICAL_ROSTER_AUDIT_BASELINE_CHANGED\')\n    eligible = expected - excluded\n    return eligible, dict(basis=\'AUDITED_MISSING_ONLY_NON_PIT\', audit_sha256=spec[\'sha256\'],\n                          audited_exclusions=len(excluded), historical_pit_certified=False)\n\n\ndef period_market_roster(root, current_roster, periods):\n    """Choose the nearest available archived company snapshot for each period.\n\n    Pre-archive periods use the earliest verifiable snapshot; NEVER claim this\n    recreates historical listing status or is strict PIT.  A new current roster\n    must not lower 2022/2025 coverage merely because companies IPO in 2027.\n    """\n    history=[]\n    for folder in sorted((Path(root)/\'data\').glob(\'20??-??\')):\n        manifests=list(folder.glob(\'manifest-r[0-9]*.json\'))\n        if not manifests:continue\n        chosen=max(manifests,key=lambda p:int(re.fullmatch(r\'manifest-r(\\d+)\\.json\',p.name).group(1)))\n        revision=json.loads(chosen.read_text(\'utf-8\')).get(\'revision\')\n        fp=folder/f\'universe-{revision}.csv\'\n        if not fp.is_file():continue\n        frame=pd.read_csv(fp,dtype={\'symbol\':str,\'market\':str})\n        pools={m:set(frame.loc[frame[\'market\'].eq(m),\'symbol\']) for m in MARKETS}\n        history.append((folder.name,pools))\n    if not history:\n        pools={m:{s for s,r in current_roster.items() if r[\'market\']==m} for m in MARKETS}\n        return {p:pools for p in periods}\n    output={}\n    for period in periods:\n        month=period\n        if \'Q\' in month:month=f\'{period[:4]}-{int(period[-1])*3:02d}\'\n        candidates=[item for item in history if item[0]<=month]\n        output[period]=(candidates[-1] if candidates else history[0])[1]\n    return output\n\n\ndef report(batch, tasks, old, audit):\n    rows = all_rows(batch,tasks,old)\n    present_index={}\n    for row in rows:\n        kind=row[\'dataset\']\n        present_index.setdefault((kind,row[\'market\'],period_of(row,kind)),set()).add(row[\'symbol\'])\n    gaps,coverage,summary = [],[],[]\n    cohorts=period_market_roster(batch.root,batch.roster,{t[\'period\'] for t in tasks})\n    for task in tasks:\n        kind,market,period = task[\'kind\'],task[\'market\'],task[\'period\']\n        present = present_index.get((kind,market,period),set())\n        raw_expected = cohorts[period][market]\n        expected, eligibility = audited_eligible_roster(batch.root, kind, market, period, raw_expected)\n        # An exclusion is lawful only for truly absent historical values; fail on conflict.\n        if eligibility is not None and len(present & set(raw_expected)) != len(present & expected):\n            raise RuntimeError(\'HISTORICAL_ROSTER_AUDIT_OBSERVED_CONFLICT\')\n        ratio = len(present & expected)/len(expected) if expected else 0\n        rec = batch.load(task)\n        coverage.append(dict(dataset=kind,market=market,period=period,expected=len(expected),\n                             observed=len(present & expected),coverage=ratio,\n                             official_batch_saved=bool(rec),status=\'PRESENT\' if present else \'MISSING\',\n                             basis=eligibility[\'basis\'] if eligibility else \'CURRENT_ROSTER_NOT_HISTORICAL_PIT\'))\n        for symbol in sorted(expected-present):\n            gaps.append(dict(dataset=kind,market=market,period=period,symbol=symbol,\n                             reason=\'MISSING_OR_PRE_IPO_OR_NOT_ANNOUNCED\'))\n    for kind in sorted({t[\'kind\'] for t in tasks}):\n        group=[r for r in coverage if r[\'dataset\']==kind]\n        summary.append(dict(dataset=kind,batches_planned=len(group),official_saved=sum(r[\'official_batch_saved\'] for r in group),\n                            missing_symbol_periods=sum(r[\'expected\']-r[\'observed\'] for r in group)))\n    for kind in sorted({r[\'task\'][\'kind\'] for r in old}-{t[\'kind\'] for t in tasks}):\n        summary.append(dict(dataset=kind,batches_planned=None,official_saved=0,\n                            missing_symbol_periods=None,legacy_rows=sum(len(r[\'rows\']) for r in old if r[\'task\'][\'kind\']==kind)))\n    dest=batch.root/\'reports\'\n    for name,data in [(\'ohlcv_audit\',audit),(\'collection_summary\',summary),(\'monthly_quarterly_coverage\',coverage),\n                      (\'missing_symbol_periods\',gaps),(\'source_errors\',batch.errors),\n                      (\'role_readiness\',[dict(role=r,training_ready=False,status=\'INTERMEDIATE_REPORT_NOT_ASSESSED\')\n                                         for r in [\'Quant\',\'Risk\',\'Fundamental\',\'Valuation\']])]:\n        atomic(dest/(name+\'.csv\'),pd.DataFrame(data).to_csv(index=False).encode(\'utf-8-sig\'))\n    return pd.DataFrame(summary),pd.DataFrame(coverage),gaps,dest\n\ndef export(batch,tasks,old):\n    """Official and legacy sources stay separate, with immutable month partitions."""\n    unique_rows={sha(encode(row)):row for row in all_rows(batch,tasks,old)}\n    rows=[unique_rows[key] for key in sorted(unique_rows)]\n    generation=sha(encode(rows))[:16]\n    dest=batch.root/\'_derived\'/\'official-first-v2\'/generation\n    manifest=dest/\'manifest.json\'\n    if manifest.exists():\n        obj=json.loads(manifest.read_text(\'utf-8\'))\n        for name,digest in obj[\'sha256\'].items():\n            if sha((dest/name).read_bytes()) != digest:\n                raise ValueError(\'EXPORT_CHECKSUM_MISMATCH\')\n        return dest\n    partitions={}\n    for row in rows:\n        key=(row[\'partition_month\'],row[\'dataset\'],row[\'source_id\'])\n        partitions.setdefault(key,[]).append(row)\n    hashes={}\n    for (month,kind,source),values in sorted(partitions.items()):\n        name=f\'{month}/{kind}-{source}.csv.gz\'\n        # Remove exact duplicate rows shared by old plans without merging conflicting sources.\n        unique={sha(encode(r)):r for r in values}\n        body=gzip.compress(pd.DataFrame(unique.values()).to_csv(index=False).encode(\'utf-8\'),mtime=0)\n        atomic(dest/name,body)\n        hashes[name]=sha(body)\n    atomic(manifest,encode(dict(version=VERSION,sha256=hashes,historical_pit=\'NOT_VERIFIED\',\n                               publication_allowed=False,official_source_priority=True)))\n    return dest\n\nclass ReuseCollector(Collector):\n    def reuse(self, old):\n        existing=self.records()\n        reused=0\n        by_key={}\n        for rec in old:\n            t=rec[\'task\']\n            by_key.setdefault((t[\'kind\'],t[\'symbol\']),[]).append(rec)\n        for i,t in enumerate(self.tasks):\n            if t[\'id\'] in existing:\n                continue\n            for rec in by_key.get((t[\'kind\'],t[\'symbol\']),[]):\n                prev=rec[\'task\']\n                if prev[\'start\'] <= t[\'start\'] and prev[\'end\'] >= t[\'end\']:\n                    accepted,rejected=normalise(t[\'kind\'],rec[\'raw_rows\'],t,self.roster,rec[\'observed_at\'])\n                    # Range filtering is not a provider schema error.\n                    rejected=[r for r in rejected if r[\'reason\']!=\'OUTSIDE_REQUEST_RANGE\']\n                    if accepted and not rejected:\n                        copied=dict(rec,task=t,rows=accepted,rejected=[],reused_from=\'prior_plan\')\n                        self.save(i,copied)\n                        reused+=1\n                        break\n        return reused\n\ndef fallback_tasks(roster,end_month,gaps,batch,tasks):\n    """Fallback only after all official adapters have been attempted this run or saved."""\n    attempted={(e[\'task\'][\'kind\'],e[\'task\'][\'market\'],e[\'task\'][\'period\']) for e in batch.errors}\n    eligible={(t[\'kind\'],t[\'market\'],t[\'period\']) for t in tasks if batch.load(t) or\n              (t[\'kind\'],t[\'market\'],t[\'period\']) in attempted}\n    output=[]\n    windows={}\n    for g in gaps:\n        if (g[\'dataset\'],g[\'market\'],g[\'period\']) not in eligible:\n            continue\n        p=pd.Period(g[\'period\'],\'Q\' if g[\'dataset\'] in FINANCIAL else \'M\')\n        # Revenue API date is publication/provider date; include one following month.\n        end=(p.asfreq(\'M\',\'end\')+1).end_time.date() if g[\'dataset\']==\'revenue\' else p.end_time.date()\n        key=(g[\'dataset\'],g[\'symbol\'])\n        start=str(p.start_time.date())\n        end=str(end)\n        previous=windows.get(key,(start,end))\n        windows[key]=(min(start,previous[0]),max(end,previous[1]))\n    for (kind,symbol),(start,end) in sorted(windows.items()):\n        t=dict(kind=kind,symbol=symbol,start=start,end=end)\n        t[\'id\']=sha(encode(t))[:24]\n        output.append(t)\n    extras=[t for t in build_tasks(roster,end_month) if t[\'kind\'] not in {\'revenue\',\'valuation\',*FINANCIAL}]\n    # Shared snapshots first, then actual gaps; per-stock events follow.\n    return [t for t in extras if t[\'symbol\'] in (\'*\',\'TAIEX\',\'TPEx\')]+output+[\n        t for t in extras if t[\'symbol\'] not in (\'*\',\'TAIEX\',\'TPEx\')]\n\n\ndef due_for_collection(kind, period, as_of):\n    """Grace period prevents reporting not-yet-published financial/revenue as missing."""\n    if kind in FINANCIAL:\n        return (as_of - pd.Period(period, freq=\'Q\').end_time.date()).days >= 50\n    if kind == \'revenue\':\n        return (as_of - pd.Period(period, freq=\'M\').end_time.date()).days >= 15\n    return (as_of - pd.Period(period, freq=\'M\').end_time.date()).days >= 0\n\n\ndef targeted_missing_tasks(batch, tasks, *, as_of=None, covered=None):\n    """Bounded dynamic backlog for all datasets/markets in each newly closed month.\n\n    Official checkpoint already saved -> do not fetch again even if some individual\n    companies lack data: those require targeted provider fallback or accepted missing.\n    """\n    as_of = as_of or date.today()\n    covered = covered or set()\n    priority = {\'valuation\': 0, \'revenue\': 1, \'income\': 2, \'balance\': 3, \'cashflow\': 4}\n    output=[]\n    for t in tasks:\n        key=t[\'kind\'],t[\'market\'],t[\'period\']\n        if key in covered or not due_for_collection(t[\'kind\'],t[\'period\'],as_of):\n            continue\n        if batch.load(t) is None:\n            output.append(t)\n    # Most recent missing periods first for practical incremental service, then older gaps.\n    return sorted(output,key=lambda t:(priority.get(t[\'kind\'],9),-pd.Period(t[\'period\'],freq=(\'Q\' if t[\'kind\'] in FINANCIAL else \'M\')).ordinal,t[\'market\']))\n\ndef targeted_fallback_tasks(roster, gaps, missing_batches):\n    """One FinMind request per symbol/dataset across all missing periods.\n\n    Never broaden into global extra datasets or already-adequate markets.\n    Call limits remain enforced by ReuseCollector; leftovers persist for next run.\n    """\n    wanted={(t[\'kind\'],t[\'market\'],t[\'period\']) for t in missing_batches}\n    windows={}\n    for g in gaps:\n        if (g[\'dataset\'],g[\'market\'],g[\'period\']) not in wanted:\n            continue\n        kind,symbol,period=g[\'dataset\'],g[\'symbol\'],g[\'period\']\n        if symbol not in roster:continue\n        p=pd.Period(period, freq=\'Q\' if kind in FINANCIAL else \'M\')\n        start=str(p.start_time.date())\n        end=str(p.end_time.date())\n        old=windows.get((kind,symbol))\n        windows[(kind,symbol)]=(min(start,old[0]),max(end,old[1])) if old else (start,end)\n    items=[]\n    priority={\'valuation\':0,\'balance\':1,\'cashflow\':2,\'income\':3}\n    for (kind,symbol),(start,end) in sorted(windows.items(),key=lambda x:(priority.get(x[0][0],9),x[0][1])):\n        t={\'kind\':kind,\'symbol\':symbol,\'start\':start,\'end\':end}\n        t[\'id\']=sha(encode(t))[:24]\n        items.append(t)\n    return items\n\nTPEx_VALUATION_ADAPTER_ERRORS = frozenset({\n    \'VALUATION_UNRECOGNISED_EMPTY\',\n    \'TPEx_HTML_DATE_NOT_CONFIRMED\',\n    \'TPEx_HTML_SCHEMA_UNRECOGNISED\',\n    \'TPEx_HTML_TABLE_NOT_FOUND\',\n    \'RESPONSE_DATE_MISSING\',\n})\n\n\ndef eligible_fallback_keys(batch, tasks, covered, *, as_of):\n    """Use FinMind only for still-undercovered, due periods after official attempt.\n\n    On a *confirmed adapter-schema failure* for TPEx valuation, another twenty\n    monthly official tasks may have been skipped by the bounded failure guard.\n    These may use the bounded FinMind backup without repeating known-invalid\n    official requests. Never broaden on HTTP 401/403/429, quota or time limits.\n    """\n    attempted = {(e[\'task\'][\'kind\'],e[\'task\'][\'market\'],e[\'task\'][\'period\'])\n                 for e in batch.errors}\n    saved = {(t[\'kind\'],t[\'market\'],t[\'period\']) for t in tasks if batch.load(t) is not None}\n    adapter_failed = any(e[\'task\'][\'kind\']==\'valuation\' and\n                         e[\'task\'][\'market\']==\'tpex\' and\n                         e[\'error\'] in TPEx_VALUATION_ADAPTER_ERRORS for e in batch.errors)\n    blocked = {(e[\'task\'][\'kind\'],e[\'task\'][\'market\']) for e in batch.errors\n               if str(e[\'error\']).startswith((\'SOURCE_BLOCKED\',\'AUTH_OR_QUOTA\',\'PROVIDER_STATUS_429\'))}\n    eligible = attempted | saved\n    if adapter_failed:\n        eligible |= {(t[\'kind\'],t[\'market\'],t[\'period\']) for t in tasks\n                     if t[\'kind\']==\'valuation\' and t[\'market\']==\'tpex\'}\n    return {k for k in eligible if k not in covered and\n            (k[0],k[1]) not in blocked and due_for_collection(k[0],k[2],as_of)}\n', 'yahoo_fallback.py': '"""Yahoo Finance optional supplemental collector for research, not canonical publication."""\nfrom __future__ import annotations\nimport gzip\nimport json\nfrom pathlib import Path\nimport pandas as pd\nfrom research_collection import atomic, now, encode\n\n\ndef supplement(root, roster, gaps, *, max_symbols=20, include_financials=True):\n    import yfinance as yf\n    root=Path(root)\n    seen=set()\n    output=[]\n    kinds={\'income\':\'quarterly_income_stmt\',\'balance\':\'quarterly_balance_sheet\',\'cashflow\':\'quarterly_cash_flow\'}\n    finance_gaps={(g[\'dataset\'],g[\'symbol\'],g[\'period\']) for g in gaps if g[\'dataset\'] in kinds}\n    price_gaps={(g[\'symbol\'],g[\'period\']) for g in gaps if g[\'dataset\']==\'ohlcv\'}\n    for symbol, period in sorted({(s,p) for k,s,p in finance_gaps} | price_gaps):\n        if symbol in seen or len(seen)>=max_symbols:\n            continue\n        if symbol not in roster:\n            continue\n        # Skip existing success checkpoints instead of refetching the same financial rows.\n        if all((root/\'data\'/f\'{int(q[:4])}-{int(q[-1])*3:02d}\'/f\'yahoo-{kind}-{symbol}.json.gz\').exists()\n               for kind,s,q in finance_gaps if s==symbol):\n            continue\n        seen.add(symbol)\n        market=roster[symbol][\'market\']\n        ticker=f"{symbol}.{\'TW\' if market==\'twse\' else \'TWO\'}"\n        try:\n            tk=yf.Ticker(ticker)\n            if include_financials:\n                for kind,attr in kinds.items():\n                    quarters={p for k,s,p in finance_gaps if k==kind and s==symbol}\n                    if not quarters:\n                        continue\n                    frame=getattr(tk,attr,None)\n                    if frame is None or frame.empty:\n                        continue\n                    for col in frame.columns:\n                        stamp=pd.Timestamp(col)\n                        q=f\'{stamp.year}Q{(stamp.month-1)//3+1}\'\n                        if q not in quarters:\n                            continue\n                        destination=root/\'data\'/stamp.strftime(\'%Y-%m\')/f\'yahoo-{kind}-{symbol}.json.gz\'\n                        if destination.exists():\n                            continue\n                        out=[]\n                        for field,value in frame[col].items():\n                            numeric=pd.to_numeric(value,errors=\'coerce\')\n                            if pd.isna(numeric):\n                                continue\n                            out.append(dict(symbol=symbol,market=market,dataset=kind,\n                                            fiscal_quarter=q,partition_month=stamp.strftime(\'%Y-%m\'),\n                                            item=str(field),value=float(numeric),source_id=\'yahoo\',\n                                            unit_basis=\'YAHOO_PROVIDER_UNVERIFIED\',\n                                            period_basis=\'YAHOO_STATEMENT_COLUMN\',\n                                            historical_pit=\'NOT_VERIFIED\',publication_allowed=False,\n                                            observed_at=now()))\n                        if out:\n                            folder=root/\'data\'/stamp.strftime(\'%Y-%m\')\n                            atomic(destination,gzip.compress(encode(out),mtime=0))\n                            output.append(dict(symbol=symbol,kind=kind,period=q,rows=len(out),status=\'COLLECTED\'))\n        except Exception as exc:\n            output.append(dict(symbol=symbol,status=\'ERROR\',reason=type(exc).__name__))\n    return pd.DataFrame(output)\n', 'monthly_folder_io.py': '"""Read Janus Colab-first immutable YYYY-MM research folders with content hashes."""\nfrom __future__ import annotations\n\nimport gzip\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport pandas as pd\n\n\n# First-stage research admission by ordinary-stock *snapshot* coverage.\n# This is not historical PIT certification or a model promotion decision.\nMIN_RESEARCH_COVERAGE = {"twse": 0.90, "tpex": 0.70}\n\n\ndef assess_monthly_ordinary_stock_coverage(quality: dict, universe: pd.DataFrame,\n                                           bars: pd.DataFrame, manifest: dict | None = None) -> dict:\n    """Check separately by market without inventing missing stocks or prices.\n\n    Denominator: current ordinary-stock roster snapshot. Numerator: distinct\n    symbols with >=1 observed bar in the month. Historical delistings remain\n    unknown, so accepted data stays research-only / non-PIT.\n    """\n    report = {}\n    counts = quality.get(\'market_counts\')\n    if not isinstance(counts, dict):\n        raise ValueError(\'monthly market coverage evidence missing\')\n    for market, minimum in MIN_RESEARCH_COVERAGE.items():\n        denominator = int(universe[\'market\'].eq(market).sum())\n        observed = int(bars.loc[bars[\'market\'].eq(market), \'symbol\'].nunique())\n        if denominator <= 0:\n            raise ValueError(f\'{market} ordinary-stock roster empty\')\n        fraction = observed / denominator\n        saved = counts.get(market)\n        if (not isinstance(saved, dict)\n                or saved.get(\'snapshot_symbols\') != denominator\n                or saved.get(\'symbols_with_bars\') != observed\n                or abs(float(saved.get(\'observed_ratio\', -1)) - fraction) > 1e-9):\n            raise ValueError(f\'{market} published coverage evidence disagrees with OHLCV/roster\')\n        stock_day_ratio = None\n        if manifest is not None:\n            statuses = manifest.get(\'source_daily_sha256\', [])\n            if not statuses or len(statuses) % 2:\n                raise ValueError(\'month market-session provenance invalid\')\n            half = len(statuses) // 2\n            part = statuses[:half] if market == \'twse\' else statuses[half:]\n            sessions = sum(row.get(\'status\') == \'COLLECTED\' for row in part)\n            if sessions <= 0:\n                raise ValueError(f\'{market} trading sessions not evidenced\')\n            actual_bars = int(bars[\'market\'].eq(market).sum())\n            stock_day_ratio = actual_bars / (sessions * denominator)\n        report[market] = {\'observed_symbols\': observed, \'snapshot_symbols\': denominator,\n                          \'observed_ratio\': fraction, \'minimum\': minimum,\n                          \'stock_day_ratio\': stock_day_ratio,\n                          \'passes\': fraction >= minimum\n                          and (stock_day_ratio is None or stock_day_ratio >= minimum)}\n    if not all(x[\'passes\'] for x in report.values()):\n        raise ValueError(\'ordinary-stock research coverage below 90% TWSE / 70% TPEx\')\n    return {\'status\': \'RESEARCH_COVERAGE_ACCEPTED_NOT_PIT_CERTIFIED\',\n            \'basis\': \'CURRENT_MASTER_ONLY\', \'markets\': report}\n\n\ndef load_monthly_folders(folders: Iterable[str | Path]) -> tuple[pd.DataFrame, list[dict]]:\n    frames = []\n    provenance = []\n    seen_month = set()\n    for directory in folders:\n        directory = Path(directory)\n        if not directory.is_dir() or not directory.name.startswith(\'20\'):\n            raise ValueError(\'expected YYYY-MM research folder\')\n        manifests = list(directory.glob(\'manifest-r[0-9]*.json\'))\n        if not manifests:\n            raise ValueError(\'research month has no finalized manifest\')\n        # Highest finalized revision is explicit; a newer revision never rewrites r1.\n        manifest_path = max(manifests, key=lambda p:int(re.fullmatch(r\'manifest-r(\\d+)\\.json\',p.name).group(1)))\n        manifest = json.loads(manifest_path.read_text(\'utf-8\'))\n        month, revision = manifest.get(\'month\'), manifest.get(\'revision\')\n        if directory.name != month or month in seen_month:\n            raise ValueError(\'duplicate month or mismatched directory\')\n        seen_month.add(month)\n        if manifest.get(\'kind\') != \'COLAB_RESEARCH_PROBE\' or manifest.get(\'publication_allowed\') is not False:\n            raise ValueError(\'unexpected research manifest\')\n        declared = manifest.get(\'sha256\', {})\n        required = (f\'ohlcv-{revision}.csv.gz\', f\'universe-{revision}.csv\', f\'quality-{revision}.json\')\n        for name in required:\n            if name not in declared or \'/\' in name or \'..\' in name:\n                raise ValueError(\'mandatory research hash absent\')\n            f = directory / name\n            if not f.is_file() or hashlib.sha256(f.read_bytes()).hexdigest() != declared[name]:\n                raise ValueError(f\'SHA256 mismatch: {name}\')\n        quality = json.loads((directory / required[2]).read_text(\'utf-8\'))\n        if quality.get(\'status\') not in (\'PROBE\', \'PARTIAL\', \'PASS\'):\n            raise ValueError(\'unrecognized research quality status\')\n        universe = pd.read_csv(directory / required[1], dtype={\'symbol\': \'string\', \'market\': \'string\'})\n        if universe.duplicated([\'symbol\']).any() or not universe[\'market\'].isin([\'twse\', \'tpex\']).all():\n            raise ValueError(\'invalid research company roster\')\n        with gzip.open(directory / required[0], \'rb\') as stream:\n            df = pd.read_csv(stream, dtype={\'symbol\': \'string\', \'market\': \'string\'})\n        cols = {\'symbol\', \'market\', \'trade_date\', \'close\', \'volume\'}\n        if not cols.issubset(df.columns):\n            raise ValueError(\'research OHLCV fields missing\')\n        if not df.empty:\n            if not df[\'trade_date\'].str.startswith(month + \'-\').all():\n                raise ValueError(\'research OHLCV dates do not match month\')\n            match = df.merge(universe[[\'symbol\', \'market\']], on=[\'symbol\', \'market\'], how=\'left\', indicator=True)\n            if not (match[\'_merge\'] == \'both\').all():\n                raise ValueError(\'research row outside official company roster\')\n        coverage = assess_monthly_ordinary_stock_coverage(quality, universe, df, manifest)\n        statuses = manifest.get(\'source_daily_sha256\', [])\n        if not statuses or len(statuses) % 2:\n            raise ValueError(\'monthly official session evidence missing\')\n        half = len(statuses) // 2\n        market_sessions = {}\n        for market, part in ((\'twse\', statuses[:half]), (\'tpex\', statuses[half:])):\n            dates = [entry[\'day\'] for entry in part]\n            if len(dates) != len(set(dates)) or dates != sorted(dates):\n                raise ValueError(\'invalid official trading-day order\')\n            market_sessions[market] = [entry[\'day\'] for entry in part\n                                       if entry.get(\'status\') == \'COLLECTED\']\n        df[\'source_month\'] = month\n        df[\'source_archive_sha256\'] = hashlib.sha256(manifest_path.read_bytes()).hexdigest()\n        frames.append(df)\n        provenance.append({\'archive\': str(directory), \'month\': month,\n                           \'sha256\': df[\'source_archive_sha256\'].iloc[0] if not df.empty else hashlib.sha256(manifest_path.read_bytes()).hexdigest(),\n                           \'quality_status\': quality[\'status\'],\n                           \'historical_survivorship\': quality.get(\'historical_survivorship\', \'UNKNOWN\'),\n                           \'research_coverage\': coverage,\n                           \'market_sessions\': market_sessions})\n    if not frames:\n        raise ValueError(\'no monthly research folders\')\n    combined = pd.concat(frames, ignore_index=True)\n    combined[\'trade_date\'] = pd.to_datetime(combined[\'trade_date\'], format=\'%Y-%m-%d\', errors=\'raise\')\n    if combined.duplicated([\'symbol\', \'trade_date\']).any():\n        raise ValueError(\'duplicate symbol/trade_date in monthly research data\')\n    return combined, provenance\n'}
for filename,content in SOURCES.items():
    (CODE_DIR/filename).write_text(content,encoding='utf-8')
if str(CODE_DIR) not in sys.path:sys.path.insert(0,str(CODE_DIR))
# Running a newer Notebook in an existing Colab kernel must not reuse old code.
for module_name in [Path(n).stem for n in SOURCES]:
    sys.modules.pop(module_name,None)
importlib.invalidate_caches()
from research_collection import audit_ohlcv,collect_ohlcv
from official_batch import BatchCollector,make_tasks,report,all_rows,legacy_records,ReuseCollector,targeted_missing_tasks,targeted_fallback_tasks,due_for_collection,eligible_fallback_keys
from canonical_v3 import build_canonical
from quality_gate_zh import assess,show_zh_dashboard
import pandas as pd
from IPython.display import display


## 3.1．僅重驗既有 Canonical（免重抓／免重建資料包）

將前面的 `RECHECK_ONLY=True`，執行所有儲存格即可對原 immutable Parquet release 逐檔執行 SHA256、重新計算兩市場覆蓋率並更新中文報表。不會補收、重建或修改 canonical 指標。預設 False 保留長期增量採集行為。


In [ ]:
# 單獨重驗收：設定 RECHECK_ONLY=True 後「全部執行」僅會使用 Drive 現有月檔／canonical。
if RECHECK_ONLY:
    audit,roster=audit_ohlcv(ROOT,START_MONTH,END_MONTH)
    if not roster or any(a['status'] in ('FAILED_VALIDATION','INVALID_ROWS') for a in audit):
        raise RuntimeError('既有OHLCV名冊或月檔驗證失敗，不可重驗收')
    tasks=make_tasks(END_MONTH,
        revenue_months=int(pd.Period(END_MONTH,'M').ordinal-pd.Period(REVENUE_FIRST_MONTH,'M').ordinal+1),
        valuation_months=int(pd.Period(END_MONTH,'M').ordinal-pd.Period(VALUATION_FIRST_MONTH,'M').ordinal+1),
        financial_quarters=16, revenue_first=REVENUE_FIRST_MONTH,
        valuation_first=VALUATION_FIRST_MONTH, financial_first=FINANCIAL_FIRST_QUARTER)
    batch=BatchCollector(ROOT,roster,max_requests=0,max_minutes=0,interval=0)
    pointer=ROOT/'canonical'/'CURRENT.json'
    if not pointer.is_file():raise FileNotFoundError('CURRENT.json 不存在')
    release_id=json.loads(pointer.read_text('utf-8')).get('release','')
    if not release_id.startswith('release-') or '/' in release_id:
        raise RuntimeError('不合法的 immutable release 名稱')
    release=ROOT/'canonical'/release_id
    manifest=json.loads((release/'manifest.json').read_text('utf-8'))
    if manifest.get('format')!='parquet':raise RuntimeError('只接受已封存的 Parquet release')
    gate=assess(ROOT,release,manifest,roster,tasks,audit=audit,batch=batch,
                as_of=as_of,validate_hashes=True)
    show_zh_dashboard(gate)
    print('僅重驗資料品質；沿用已驗證的 CURRENT：',release.name,
          '｜本輪未補收、未生成新 Parquet release')


## 4．檢查歷史資料並補收新月份 OHLCV
原有 `r1` 封存月檔不重抓；新月份按年份名冊 revision 收集，限額不足時保存每個已取得交易日的 checkpoint，下次自動續跑。年初跨期不會從頭重抓 2025 年。


In [ ]:
if not RECHECK_ONLY and not FACTOR_ONLY:
    audit,roster=audit_ohlcv(ROOT,START_MONTH,END_MONTH)
    display(pd.DataFrame(audit).drop(columns=['markets'],errors='ignore'))
    if any(x['status'] in ('FAILED_VALIDATION','INVALID_ROWS') for x in audit):
        raise RuntimeError('既有月檔驗證失敗，不可直接重抓／覆蓋；請查核報告')
    missing_ohlcv=[x['month'] for x in audit if x['status']=='MISSING_OR_CHECKPOINT']
    if missing_ohlcv:
        first_missing=missing_ohlcv[0]
        # Reuse an unfinished revision (even across Dec/Jan), otherwise refresh the
        # ordinary-stock universe at most once per execution year for new months.
        checkpoint_root=ROOT/'data'/first_missing/'_checkpoint'
        unfinished=sorted([p.name for p in checkpoint_root.glob('r[0-9]*') if p.is_dir()],
                          key=lambda v:int(v[1:])) if checkpoint_root.exists() else []
        revision=unfinished[-1] if unfinished else f'r{max(2,as_of.year-2024)}'
        print(f'尚缺行情月：{len(missing_ohlcv)}；從 {first_missing} 續收，名冊修訂版 {revision}')
        outcome=collect_ohlcv(ROOT/'data',first_missing,END_MONTH,revision,
                              max_requests=OHLCV_MAX_REQUESTS,min_interval=OHLCV_INTERVAL_SECONDS,
                              max_minutes=OHLCV_MAX_MINUTES)
        display(pd.DataFrame(outcome))
        audit,roster=audit_ohlcv(ROOT,START_MONTH,END_MONTH)
    else:
        print('所有已結束月份 OHLCV 均已封存，無須下載')
    if any(x['status'] in ('FAILED_VALIDATION','INVALID_ROWS') for x in audit):
        raise RuntimeError('行情/名冊驗證異常，不更新 CURRENT')
    if not roster:
        raise RuntimeError('找不到有證據的普通股名冊，不能評估覆蓋率')
    old=legacy_records(ROOT)
    print('有效普通股名冊：',len(roster),'｜FinMind 舊資料批次：',len(old),'｜行情缺月：',len([x for x in audit if x['status']!='VERIFIED_RESEARCH']))


## 5．全資料類型、兩市場自動掃描與官方定向補收
每次自動建立所有新期別和歷史缺口清單：月營收（公告後15天）、上市／上櫃 PE/PB、季損益／資產負債／現金流（季度終了50天）。

只取未達上市90%／上櫃70%門檻且尚無成功官方批次的期別。保留原始回應、上限與可續跑狀態；已成功的批次不重抓。


In [ ]:
if not RECHECK_ONLY and not FACTOR_ONLY:
    tasks=make_tasks(END_MONTH,
                     revenue_months=int(pd.Period(END_MONTH,'M').ordinal-pd.Period(REVENUE_FIRST_MONTH,'M').ordinal+1),
                     valuation_months=int(pd.Period(END_MONTH,'M').ordinal-pd.Period(VALUATION_FIRST_MONTH,'M').ordinal+1),
                     financial_quarters=16,
                     revenue_first=REVENUE_FIRST_MONTH,valuation_first=VALUATION_FIRST_MONTH,
                     financial_first=FINANCIAL_FIRST_QUARTER)
    batch=BatchCollector(ROOT,roster,max_requests=OFFICIAL_MAX_REQUESTS,
                         max_minutes=OFFICIAL_MAX_MINUTES,interval=OFFICIAL_INTERVAL_SECONDS)
    pre_summary,pre_coverage,_,_=report(batch,tasks,old,audit)
    covered={(r['dataset'],r['market'],r['period']) for r in pre_coverage.to_dict('records')
             if r['coverage'] >= (0.9 if r['market']=='twse' else 0.7)}
    # A prior release may already pass due to FinMind or Yahoo, not visible in official-only report.
    from quality_gate_zh import _read_actual,THRESHOLDS
    pointer=ROOT/'canonical'/'CURRENT.json'
    if pointer.exists():
        p=json.loads(pointer.read_text('utf-8')).get('release','')
        prior_release=ROOT/'canonical'/p
        if p and prior_release.is_dir():
            for key,symbols in _read_actual(prior_release).items():
                if key[1] in THRESHOLDS and len(symbols & {s for s,r in roster.items() if r['market']==key[1]}) >= THRESHOLDS[key[1]]*sum(r['market']==key[1] for r in roster.values()):
                    covered.add(key)
    targets=targeted_missing_tasks(batch,tasks,as_of=as_of,covered=covered)
    print(f'官方歷史/當期總檢查 {len(tasks)} 批；本輪真正未達門檻、且官方未存成功 {len(targets)} 批')
    if targets:display(pd.DataFrame(targets).groupby(['kind','market']).size().reset_index(name='待補批數'))
    try:
        errors=batch.run(targets)
        if errors:display(pd.DataFrame(errors))
    finally:
        summary,coverage,gaps,report_dir=report(batch,tasks,old,audit)
        print('官方補收狀態：',getattr(batch,'run_state',{}),'｜品質報告：',report_dir)


## 6．FinMind 定向補缺（必要時）
先使用 TPEx 官方「指定歷史日期」HTML 整市場表格修復缺月；官方解析證實失敗才啟用 FinMind TaiwanStockPER 股票×整段缺月合併補缺。每次請求有上限、成功結果可續跑；若官方為 403／429 等來源封鎖，不以 FinMind 繞過限制。


In [ ]:
if not RECHECK_ONLY and not FACTOR_ONLY:
    if ENABLE_FINMIND_FALLBACK:
        covered |= {(r['dataset'],r['market'],r['period']) for r in coverage.to_dict('records') if r['coverage'] >= THRESHOLDS[r['market']]}
        # Even if official saved an under-covered batch, use the actual gap list for fallback.
        missing_keys={(t['kind'],t['market'],t['period']) for t in tasks
                      if due_for_collection(t['kind'],t['period'],as_of) and
                      (t['kind'],t['market'],t['period']) not in covered}
        # When TPEx's official historical adapter has a verified schema error,
        # allow a capped FinMind fallback for all its skipped historical months.
        # Never use fallback to circumvent source 401/403/429 or rate limits.
        wanted_keys=missing_keys & eligible_fallback_keys(batch,tasks,covered,as_of=as_of)
        wanted_tasks=[t for t in tasks if (t['kind'],t['market'],t['period']) in wanted_keys]
        supplement_tasks=targeted_fallback_tasks(roster,gaps,wanted_tasks)
        print(f'FinMind 僅補需補個股／資料類型：{len(supplement_tasks)}，本次最多 {FINMIND_MAX_REQUESTS} 請求')
        if supplement_tasks:
            fallback=ReuseCollector(ROOT,supplement_tasks,roster,
                             max_requests=FINMIND_MAX_REQUESTS,max_minutes=FINMIND_MAX_MINUTES,
                             interval=FINMIND_INTERVAL_SECONDS,token=FINMIND_TOKEN)
            fallback.reuse(old)
            print(fallback.run(retry_empty=RETRY_EMPTY_FINMIND))
            old=legacy_records(ROOT)
        else:print('目前無符合定向補收條件的 FinMind 請求')
    else:print('本輪未啟用 FinMind')


## 7．更新不可變 Parquet 版本、產生繁中驗收報告
保留全歷史資料；新月份或缺資料才產生新 release。以讀回實際 Parquet 及 SHA 評估上市/上櫃各期達標率，含 OHLCV 新月缺口、尚未公告及 FinMind/Yahoo 定向建議。


In [ ]:
if not RECHECK_ONLY and not FACTOR_ONLY:
    old=legacy_records(ROOT)
    summary,coverage,gaps,report_dir=report(batch,tasks,old,audit)
    if ENABLE_YAHOO_FALLBACK:
        from yahoo_fallback import supplement
        current_covered={(r['dataset'],r['market'],r['period']) for r in coverage.to_dict('records') if r['coverage'] >= THRESHOLDS[r['market']]} | covered
        keys={(t['kind'],t['market'],t['period']) for t in tasks if t['kind'] in ('income','balance','cashflow') and due_for_collection(t['kind'],t['period'],as_of) and (t['kind'],t['market'],t['period']) not in current_covered}
        due_financial_gaps=[g for g in gaps if (g['dataset'],g['market'],g['period']) in keys]
        yahoo_report=supplement(ROOT,roster,due_financial_gaps,max_symbols=YAHOO_MAX_SYMBOLS)
        if not yahoo_report.empty:display(yahoo_report)
    release,manifest=build_canonical(ROOT,all_rows(batch,tasks,old),run_tag='official-fallback-v3.3')
    gate=assess(ROOT,release,manifest,roster,tasks,audit=audit,batch=batch,as_of=as_of,
                validate_hashes=VERIFY_ALL_PARQUET_SHA256)
    show_zh_dashboard(gate)
    print('發布研究版本：',release.name,'；資料列數：',manifest['tables'])
    print('正式研究入口：',ROOT/'canonical'/'CURRENT.json')
    print('若仍 PARTIAL，按上限在下次 Colab 重跑同一份 Notebook；不需修改日期或重抓既有批次。')


## 下次執行（包含 2027／03）
- **使用同一份 Notebook 執行全部**，台灣時間 2027 年 3 月會自動處理到 **2027-02**（3 月需等 4 月封存）。
- 有新月自動加任務；缺口按期別、資料集、上市/上櫃分別顯示；有 checkpoint 的交易日、官方批次及第三方回應不用重新抓。
- 已成功舊月不因時間窗口往前移就從 Parquet 遺失；公司名冊每新研究年份更新快照，歷史 PIT 仍維持未驗證。
- 請求數／時間上限時，保留成功成果，下次重跑；來源結構或供應商 API 變動需要維護 adapter，不保證永遠不用更新解析。
- **PASS 是研究資料與試訓准入，不代表經 OOS 驗證可正式上線。**

輸出在 `資料源/JanusDataset_v3_202610/reports/`；訓練讀取 `canonical/CURRENT.json`。

- **2026-10 上櫃估值修正**：22 個上櫃月缺批優先使用 TPEx 官方指定日期 HTML 整市場資料，嚴格核對回應日期；若確認資料格式變更，才用 FinMind 已記錄的歷史 PE/PB 範圍補缺。未實際跑新 Colab 前保持 PARTIAL。

- **2026-10 TPEx 歷史空日 v3.5**：實際官方原始 HTML 中存在資料日期正確但 `tbody` 空／`共0筆` 情況。這是日期當天無資料，不是 schema 錯誤；有界向前查詢可用交易日，並保存實際回應原檔及日期，仍未取得值者保持 PARTIAL。


## 8．四模型共用新因子：市場大盤 C1 來源探查（v3.7 候選）

**此階段只驗證來源與保存 raw，不改 `canonical/CURRENT.json`，不重抓原 OHLCV、財報與估值。**

要做四模型新因子的**第一輪小批官方市場指數探查**：在第 2 節設定 `FACTOR_ONLY=True`、`FACTOR_SOURCE_PROBE=True`、`RECHECK_ONLY=False` 後執行全部。只對 2025-01 和最近已完成月份分別檢查 TWSE／TPEx（單輪最多 4 請求）。既有 00 模型資料不因探查重建；只有來源 schema/date 檢查成功才給 `SCHEMA_VERIFIED`。

TWSE 檢查官方歷史月份 JSON 欄位、日期和 OHLC；TPEx 第一輪僅保留官方候選歷史頁的 raw、status 與來源證據，**還未有歷史 API schema 讀回，不會輸出假正規化結果**。403、429 等立即停止；再次跑已有 `SCHEMA_VERIFIED`／`RAW_CAPTURED_SCHEMA_PENDING` 的期別會跳過。新證據寫 `derived/factor_inputs/probes/`，不會冒充新的 immutable Parquet release。

原本一般增量採集時請讓 `FACTOR_ONLY=False`、`FACTOR_SOURCE_PROBE=False`，完全沿用前面 v3.6 流程。下一階段 C2 全期間補收與產業歷史分類，必須先依此 C1 真實 raw 檢查補完 adapter，見 GitHub `doc/spec/d07-four-model-factor-colab-intake.md`；**不要把 C1 視為所有因子完成。**


In [ ]:
# 版本化、受限且可讀回的純研究市場指數來源探查，不修改 canonical。
if FACTOR_SOURCE_PROBE:
    if not FACTOR_ONLY:
        raise RuntimeError('為免在舊整批收集中混入來源探查，請先設定 FACTOR_ONLY=True')
    # 本 cell 嵌入經本地 targeted tests 的 source/provenance helper；不需要 clone GitHub。
    import types
    _factor_probe_code = '"""C1 only: bounded official market-index SOURCE probe for four Janus research models.\nThis never changes canonical/CURRENT.json or publication_allowed.\n"""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport re\nimport time\nfrom datetime import date, datetime, timezone\nfrom pathlib import Path\nfrom urllib.parse import urlencode\n\nimport requests\n\nSOURCES = {\n    "twse_market_index": {\n        "url": "https://www.twse.com.tw/rwd/zh/indicesReport/MI_5MINS_HIST",\n        "authority": "https://data.gov.tw/dataset/11755",\n        "market": "twse",\n    },\n    "tpex_market_index": {\n        "url": "https://www.tpex.org.tw/www/zh-tw/mainboard/trading/info/daily-indices",\n        "authority": "https://data.gov.tw/dataset/11391",\n        "market": "tpex",\n    },\n}\n\n\ndef _month_checked(value: str) -> str:\n    if not re.fullmatch(r"20\\d{2}-(0[1-9]|1[0-2])", value):\n        raise ValueError("invalid closed month")\n    return value\n\n\ndef _roc_date(v: str) -> str:\n    s = str(v).strip().replace("年", "/").replace("月", "/").replace("日", "")\n    m = re.fullmatch(r"(\\d{2,4})/(\\d{1,2})/(\\d{1,2})", s)\n    if not m:\n        raise ValueError("invalid source date: " + s)\n    year = int(m[1]); year = year + 1911 if year < 1911 else year\n    return date(year, int(m[2]), int(m[3])).isoformat()\n\n\ndef _number(v) -> float:\n    if not isinstance(v, (int, float, str)):\n        raise ValueError("bad number type")\n    raw = str(v).replace(",", "").strip()\n    if not re.fullmatch(r"\\d+(\\.\\d+)?", raw):\n        raise ValueError("nonpositive, blank, or malformed index number")\n    n = float(raw)\n    if not (0 < n < 1e8):\n        raise ValueError("out of bounds index")\n    return n\n\n\ndef _normalize_twse_index(payload: dict, month: str) -> list[dict]:\n    if not isinstance(payload, dict) or str(payload.get("stat", "")).upper() != "OK":\n        raise ValueError("non-OK TWSE response")\n    fields, rows = payload.get("fields"), payload.get("data")\n    if not isinstance(fields, list) or not isinstance(rows, list) or not rows:\n        raise ValueError("TWSE schema or rows missing")\n    want = {"open": "開盤指數", "high": "最高指數", "low": "最低指數", "close": "收盤指數"}\n    matches = {}\n    for key, label in want.items():\n        hits = [i for i, name in enumerate(fields) if isinstance(name, str) and label in name]\n        if len(hits) != 1:\n            raise ValueError("index field not unique: " + key)\n        matches[key] = hits[0]\n    dates = [i for i, field in enumerate(fields) if isinstance(field, str) and "日期" in field]\n    if len(dates) != 1:\n        raise ValueError("date field not unique")\n    out = []\n    for row in rows:\n        if not isinstance(row, list) or len(row) != len(fields):\n            raise ValueError("malformed TWSE row")\n        d = _roc_date(row[dates[0]])\n        if not d.startswith(month + "-"):\n            raise ValueError("source month mismatch: " + d)\n        rec = {"trade_date": d, "market": "twse", "index_id": "TAIEX"}\n        rec.update({col: _number(row[pos]) for col, pos in matches.items()})\n        if not rec["low"] <= rec["open"] <= rec["high"] or not rec["low"] <= rec["close"] <= rec["high"]:\n            raise ValueError("invalid OHLC range")\n        out.append(rec)\n    values = [d["trade_date"] for d in out]\n    if len(values) != len(set(values)) or values != sorted(values):\n        raise ValueError("duplicate or unsorted market dates")\n    return out\n\n\ndef _url(source: str, month: str) -> str:\n    year, mm = map(int, _month_checked(month).split("-"))\n    info = SOURCES[source]\n    if source == "twse_market_index":\n        params = {"response": "json", "date": f"{year:04d}{mm:02d}01"}\n    else:\n        # Candidate historical web query: COLAB source verification required.\n        params = {"date": f"{year-1911}/{mm:02d}/01", "response": "json"}\n    return info["url"] + "?" + urlencode(params)\n\n\ndef run_market_probes(root: Path, end_month: str, *, max_requests: int = 4,\n                      interval: float = 2.0, fetch=None) -> list[dict]:\n    """Never backfills unseen months without a separate schema-verified implementation.\n\n    Default probes both markets at the oldest fixed month and newest closed month.\n    Do not interpret captured raw TPEx data as verified normalized index rows.\n    """\n    end_month = _month_checked(end_month)\n    if end_month > date.today().strftime("%Y-%m"):\n        raise ValueError("future month not allowed")\n    if max_requests < 1 or max_requests > 12 or interval < 0:\n        raise ValueError("unbounded probe settings")\n    base = Path(root) / "derived" / "factor_inputs" / "probes"\n    months = list(dict.fromkeys(["2025-01", end_month]))\n    ops = [(source, month) for month in months for source in SOURCES]\n    results = []\n    fetch = fetch or requests.get\n    count = 0\n    for source, month in ops:\n        folder = base / source / month\n        receipt_path = folder / "receipt.json"\n        if receipt_path.exists():\n            prior = json.loads(receipt_path.read_text(encoding="utf-8"))\n            if prior.get("status") in ("SCHEMA_VERIFIED", "RAW_CAPTURED_SCHEMA_PENDING"):\n                results.append({"source": source, "month": month, "status": "SKIPPED_EXISTING"})\n                continue\n        if count >= max_requests:\n            results.append({"source": source, "month": month, "status": "BUDGET_PAUSED"})\n            continue\n        url = _url(source, month)\n        count += 1\n        stamp = datetime.now(timezone.utc).isoformat()\n        try:\n            response = fetch(url, timeout=25, headers={"User-Agent": "JanusResearch/1.0 (manual-source-audit)"})\n            status_code = int(response.status_code)\n            raw = response.content\n            if len(raw) > 3_000_000:\n                raise ValueError("source response oversized")\n            sha = hashlib.sha256(raw).hexdigest()\n            folder.mkdir(parents=True, exist_ok=True)\n            raw_path = folder / f"raw-{sha}.bin"\n            if not raw_path.exists():\n                raw_path.write_bytes(raw)\n            receipt = {"source": source, "month": month, "url": url,\n                       "authority_url": SOURCES[source]["authority"],\n                       "fetched_at": stamp, "status_code": status_code,\n                       "raw_sha256": sha, "raw_file": raw_path.name,\n                       "status": "RAW_CAPTURED_SCHEMA_PENDING", "publication_allowed": False,\n                       "source_rights_for_actual_endpoint": "REQUIRES_VERIFICATION"}\n            if status_code in (401, 403, 429):\n                receipt["status"] = "SOURCE_BLOCKED"\n            elif status_code != 200:\n                receipt["status"] = "HTTP_ERROR"\n            elif source == "twse_market_index":\n                try:\n                    parsed = response.json()\n                    records = _normalize_twse_index(parsed, month)\n                    out = folder / "normalized-research.json"\n                    out.write_text(json.dumps(records, ensure_ascii=False, sort_keys=True), encoding="utf-8")\n                    receipt["status"] = "SCHEMA_VERIFIED"\n                    receipt["rows"] = len(records)\n                    receipt["normalized_sha256"] = hashlib.sha256(out.read_bytes()).hexdigest()\n                except (ValueError, KeyError, TypeError) as exc:\n                    receipt["status"] = "SCHEMA_NOT_VERIFIED"\n                    receipt["validation_error"] = str(exc)[:180]\n            else:\n                # Until schema/date semantics are confirmed against official TPEx,\n                # retain only raw bytes and provenance, never fake a market close.\n                receipt["status"] = "RAW_CAPTURED_SCHEMA_PENDING"\n            receipt_path.write_text(json.dumps(receipt, ensure_ascii=False, indent=2), encoding="utf-8")\n            results.append(receipt)\n            if receipt["status"] == "SOURCE_BLOCKED":\n                break  # do not change IP / bypass source refusal\n        except requests.RequestException as exc:\n            results.append({"source": source, "month": month,\n                            "status": "NETWORK_FAILED", "error": str(exc)[:180]})\n        if count < max_requests and interval:\n            time.sleep(interval)\n    summary_path = base / "last-run-receipt.json"\n    base.mkdir(parents=True, exist_ok=True)\n    summary_path.write_text(json.dumps({"rows": results, "requests": count,\n                     "published_to_canonical": False, "publication_allowed": False},\n                    ensure_ascii=False, indent=2), encoding="utf-8")\n    return results\n\nif __name__ == "__main__":\n    raise SystemExit("Invoke via the official Colab 00 notebook and supplied ROOT, END_MONTH")\n'
    _factor_scope={}
    exec(compile(_factor_probe_code,'colab_factor_market_probe_v37.py','exec'),_factor_scope)
    results = _factor_scope['run_market_probes'](ROOT,END_MONTH,
           max_requests=FACTOR_PROBE_MAX_REQUESTS,interval=2)
    display(pd.DataFrame(results)[['source','month','status']])
    print('C1 raw/probe 收據：',ROOT/'derived'/'factor_inputs'/'probes'/'last-run-receipt.json')
    print('此步只做 source 探查，沒有更改 CURRENT 或發布模型。')
else:
    print('新因子來源探查未啟用，原 00 收集流程不變。')


## 8．交給 ChatGPT 的明確交接（不在 Colab 貼標）
本 Notebook 的產物是 `資料源/JanusDataset_v3_202610/canonical/CURRENT.json` 及其指向的不可變 Parquet release。收資料完成後，將 CURRENT 與 referenced manifest／Parquet 的真實檔案提供給 ChatGPT 端工作目錄，不要只給 URL 或文字摘要。
在 ChatGPT 使用 `research/chatgpt/label_four_models.py --dataset-root <本地資料根> --output-root <研究輸出根>`。貼標器核對檔案 SHA256、列數、隔離 conflicts，並產生獨立 X 與候選 Y；不讀 Colab runtime 記憶體、不以抓取日倒填歷史公布日、不改 CURRENT。
**本地 notebook 靜態/單元測試通過 ≠ Colab OAuth 採集實跑 PASS；真實補收須提供 Drive release 及報表讀回證據。**
